
# **Integrated Machine Learning and Structure-Based Virtual Screening Identifies Potent Lactate Dehydrogenase A Inhibitors for Colon Cancer Therapy**
# **1. Research Pipeline:**
1. Data Preprocessing & Labeling
2. Molecular Fingerprint Generation
3. Models Training
4. Model sEvaluation
5. Virtual Screening of Compounds
6. Consensus Analysis for Docking
**Author:** Muhammad Arslan
**Date:** 02-03-2026

**Repository:** https://github.com/muhammadarslan-chemist/LDHA

In [3]:
import pandas as pd
import numpy as np

# Load the CHEMBL4835 file (LDHA bioactivity data)
df_chembl = pd.read_csv('CHEMBL4835.csv')

# Load the random compounds file (Selenium compounds from PubChem)
df_random = pd.read_csv('random.csv')

# Display basic information about both datasets
print("=" * 50)
print("CHEMBL4835 Dataset:")
print("=" * 50)
print(f"Shape: {df_chembl.shape}")
print(f"\nColumns:\n{df_chembl.columns.tolist()}")
print(f"\nFirst 5 rows:\n{df_chembl.head()}")
print(f"\nData types:\n{df_chembl.dtypes}")
print(f"\nMissing values:\n{df_chembl.isnull().sum()}")

print("\n" + "=" * 50)
print("Random Dataset (Selenium compounds):")
print("=" * 50)
print(f"Shape: {df_random.shape}")
print(f"\nColumns:\n{df_random.columns.tolist()}")
print(f"\nFirst 5 rows:\n{df_random.head()}")
print(f"\nData types:\n{df_random.dtypes}")
print(f"\nMissing values:\n{df_random.isnull().sum()}")

CHEMBL4835 Dataset:
Shape: (1174, 48)

Columns:
['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key', 'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value', 'Standard Units', 'pChEMBL Value', 'Data Validity Comment', 'Comment', 'Uo Units', 'Ligand Efficiency BEI', 'Ligand Efficiency LE', 'Ligand Efficiency LLE', 'Ligand Efficiency SEI', 'Potential Duplicate', 'Assay ChEMBL ID', 'Assay Description', 'Assay Type', 'BAO Format ID', 'BAO Label', 'Assay Organism', 'Assay Tissue ChEMBL ID', 'Assay Tissue Name', 'Assay Cell Type', 'Assay Subcellular Fraction', 'Assay Parameters', 'Assay Variant Accession', 'Assay Variant Mutation', 'Target ChEMBL ID', 'Target Name', 'Target Organism', 'Target Type', 'Document ChEMBL ID', 'Source ID', 'Source Description', 'Document Journal', 'Document Year', 'Cell ChEMBL ID', 'Properties', 'Action Type', 'Standard Text Value', 'Value']

First 5 rows:
  Molecule ChEMBL ID Molecu

In [7]:
# Step 2: Extract relevant columns from CHEMBL4835 and filter for IC50 data

# Select only essential columns for bioactivity analysis
essential_cols = [
    'Molecule ChEMBL ID', 'Smiles', 'Standard Type', 'Standard Relation', 
    'Standard Value', 'Standard Units', 'pChEMBL Value', 'Target Name'
]

df_chembl_filtered = df_chembl[essential_cols].copy()

# Filter for only IC50 data (most common bioactivity assay type)
df_ic50 = df_chembl_filtered[df_chembl_filtered['Standard Type'] == 'IC50'].copy()

print(f"Original CHEMBL4835 rows: {len(df_chembl)}")
print(f"Rows after selecting essential columns: {len(df_chembl_filtered)}")
print(f"Rows with IC50 data: {len(df_ic50)}")
print(f"Rows without IC50 data: {len(df_chembl_filtered) - len(df_ic50)}")

print("\n" + "=" * 50)
print("Standard Relation distribution in IC50 data:")
print(df_ic50['Standard Relation'].value_counts())

print("\n" + "=" * 50)
print("First 10 rows of filtered IC50 data:")
print(df_ic50.head(10))

print("\n" + "=" * 50)
print("Missing values in filtered dataset:")
print(df_ic50.isnull().sum())

Original CHEMBL4835 rows: 1174
Rows after selecting essential columns: 1174
Rows with IC50 data: 1160
Rows without IC50 data: 14

Standard Relation distribution in IC50 data:
Standard Relation
'='    908
'>'    237
'<'      1
Name: count, dtype: int64

First 10 rows of filtered IC50 data:
  Molecule ChEMBL ID                                             Smiles  \
0      CHEMBL4079370          O=C(O)c1csc(-n2nc(-c3ccc(F)c(F)c3)cc2O)n1   
1      CHEMBL3318517                  O=C1CC(c2ccccc2)CC(O)=C1Sc1ccccn1   
2      CHEMBL3318523                   O=C1CC(c2ccccc2)CC(O)=C1Sc1nncs1   
3      CHEMBL3318525                Nc1nnc(SC2=C(O)CC(c3ccccc3)CC2=O)s1   
4      CHEMBL3318527                  O=C1CC(c2ccccc2)CC(O)=C1Sc1ncccn1   
5      CHEMBL3318532            Nc1cc(N)nc(SC2=C(O)CC(c3ccccc3)CC2=O)n1   
6      CHEMBL3318533     O=C1CC(c2ccccc2)CC(O)=C1Sc1nc(O)cc(C(F)(F)F)n1   
7      CHEMBL5075865  CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=...   
8      CHEMBL3318469          O=C1C

In [10]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

def preprocess_chembl_data_publication(df):
    """
    Professional preprocessing for LDHA inhibitor data
    Following standard medicinal chemistry practices for publication
    """
    df_clean = df.copy()
    
    # 1. Filter for LDHA target only
    ldha_mask = df_clean['Target Name'].str.contains('L-lactate dehydrogenase', na=False, case=False)
    df_ldha = df_clean[ldha_mask].copy()
    print(f"Compounds targeting LDHA: {len(df_ldha)}")
    
    # 2. Keep only IC50 data
    ic50_mask = df_ldha['Standard Type'] == 'IC50'
    df_ic50 = df_ldha[ic50_mask].copy()
    print(f"Compounds with IC50 data: {len(df_ic50)}")
    
    # 3. Clean relation operators
    df_ic50['Standard Relation'] = df_ic50['Standard Relation'].astype(str).str.strip("'")
    
    # 4. Convert Standard Value to numeric
    df_ic50['IC50_nM'] = pd.to_numeric(df_ic50['Standard Value'], errors='coerce')
    df_ic50 = df_ic50.dropna(subset=['IC50_nM', 'Smiles'])
    
    # 5. Calculate pIC50 for exact values
    df_ic50['IC50_M'] = df_ic50['IC50_nM'] * 1e-9
    df_ic50['pIC50_calc'] = -np.log10(df_ic50['IC50_M'])
    
    # 6. Use pChEMBL Value if available (more reliable)
    df_ic50['pChEMBL_clean'] = pd.to_numeric(df_ic50['pChEMBL Value'], errors='coerce')
    df_ic50['pIC50'] = df_ic50['pChEMBL_clean'].fillna(df_ic50['pIC50_calc'])
    
    # 7. Activity classification based on standard thresholds
    # Active: pIC50 > 6.0 (IC50 < 1 μM)
    # Inactive: pIC50 < 5.0 (IC50 > 10 μM) 
    # Intermediate: 5.0 ≤ pIC50 ≤ 6.0 (will be excluded or handled separately)
    
    df_ic50['Activity_Label'] = 'Intermediate'
    df_ic50.loc[df_ic50['pIC50'] > 6.0, 'Activity_Label'] = 'Active'
    df_ic50.loc[df_ic50['pIC50'] < 5.0, 'Activity_Label'] = 'Inactive'
    
    # 8. Handle relational data (>, <) for publication
    # For '>' values: compound is LESS active than reported value (inactive)
    # For '<' values: compound is MORE active than reported value (active candidate)
    relation_mapping = {
        '=': 'exact',
        '>': 'inactive_bound',  # IC50 > X means less active
        '<': 'active_bound'      # IC50 < X means more active
    }
    df_ic50['Relation_Type'] = df_ic50['Standard Relation'].map(relation_mapping)
    
    # 9. For boundary cases, use conservative classification
    conservative_mask = (df_ic50['Relation_Type'] == 'inactive_bound') & (df_ic50['pIC50'] > 5.0)
    df_ic50.loc[conservative_mask, 'Activity_Label'] = 'Inactive'
    
    active_bound_mask = (df_ic50['Relation_Type'] == 'active_bound') & (df_ic50['pIC50'] < 7.0)
    df_ic50.loc[active_bound_mask, 'Activity_Label'] = 'Active'
    
    # 10. Remove duplicates (keep most potent/confident measurement)
    df_ic50 = df_ic50.sort_values(['Molecule ChEMBL ID', 'pIC50'], ascending=[True, False])
    df_unique = df_ic50.drop_duplicates(subset=['Molecule ChEMBL ID'], keep='first')
    
    # 11. Filter for clear Active/Inactive only (exclude intermediate for training)
    df_clean_labels = df_unique[df_unique['Activity_Label'].isin(['Active', 'Inactive'])].copy()
    
    # 12. Convert to binary labels
    df_clean_labels['Activity_Binary'] = (df_clean_labels['Activity_Label'] == 'Active').astype(int)
    
    # 13. Calculate molecular descriptors for quality control
    def calculate_descriptors(smiles):
        """Calculate key molecular descriptors"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                return {
                    'MolWt': Descriptors.ExactMolWt(mol),
                    'LogP': Descriptors.MolLogP(mol),
                    'HBA': Lipinski.NumHAcceptors(mol),
                    'HBD': Lipinski.NumHDonors(mol),
                    'RotBonds': Descriptors.NumRotatableBonds(mol),
                    'RingCount': Descriptors.RingCount(mol)
                }
        except:
            pass
        return {'MolWt': np.nan, 'LogP': np.nan, 'HBA': np.nan, 
                'HBD': np.nan, 'RotBonds': np.nan, 'RingCount': np.nan}
    
    # Apply descriptor calculation
    descriptors_df = df_clean_labels['Smiles'].apply(calculate_descriptors).apply(pd.Series)
    df_final = pd.concat([df_clean_labels, descriptors_df], axis=1)
    
    # 14. Apply Lipinski's Rule of Five filter (optional - comment if not needed)
    df_final['Lipinski_Violations'] = (
        (df_final['MolWt'] > 500).astype(int) +
        (df_final['LogP'] > 5).astype(int) +
        (df_final['HBA'] > 10).astype(int) +
        (df_final['HBD'] > 5).astype(int)
    )
    
    print("\n" + "=" * 50)
    print("PROCESSING SUMMARY:")
    print("=" * 50)
    print(f"Total unique compounds: {len(df_unique)}")
    print(f"Active compounds: {(df_clean_labels['Activity_Binary']==1).sum()}")
    print(f"Inactive compounds: {(df_clean_labels['Activity_Binary']==0).sum()}")
    print(f"Intermediate (excluded): {len(df_unique) - len(df_clean_labels)}")
    print(f"\nActive/Inactive ratio: {df_clean_labels['Activity_Binary'].sum()}/{len(df_clean_labels)-df_clean_labels['Activity_Binary'].sum()}")
    
    # Distribution by relation type
    print("\nActivity by Relation Type:")
    print(df_clean_labels.groupby(['Relation_Type', 'Activity_Label']).size())
    
    # Lipinski compliance
    print("\nLipinski Rule of 5:")
    print(f"Compounds with 0 violations: {(df_final['Lipinski_Violations']==0).sum()}")
    print(f"Compounds with 1+ violations: {(df_final['Lipinski_Violations']>0).sum()}")
    
    return df_final

# Process the data
print("Processing CHEMBL4835 data for publication...")
print("=" * 50)

df_chembl_final = preprocess_chembl_data_publication(df_chembl)

# Save processed data
import os
os.makedirs('data/processed', exist_ok=True)
df_chembl_final.to_csv('data/processed/chembl_ldha_processed.csv', index=False)

print("\n" + "=" * 50)
print("✓ Saved to: data/processed/chembl_ldha_processed.csv")
print("\nFinal columns in dataset:")
print(df_chembl_final.columns.tolist())
print("\nSample of processed data:")
print(df_chembl_final[['Molecule ChEMBL ID', 'Smiles', 'pIC50', 'Activity_Label', 
                        'MolWt', 'LogP', 'Lipinski_Violations']].head(10))

Processing CHEMBL4835 data for publication...
Compounds targeting LDHA: 1123
Compounds with IC50 data: 1123

PROCESSING SUMMARY:
Total unique compounds: 678
Active compounds: 291
Inactive compounds: 287
Intermediate (excluded): 100

Active/Inactive ratio: 291/287

Activity by Relation Type:
Relation_Type   Activity_Label
exact           Active            291
                Inactive          112
inactive_bound  Inactive          175
dtype: int64

Lipinski Rule of 5:
Compounds with 0 violations: 236
Compounds with 1+ violations: 342

✓ Saved to: data/processed/chembl_ldha_processed.csv

Final columns in dataset:
['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key', 'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value', 'Standard Units', 'pChEMBL Value', 'Data Validity Comment', 'Comment', 'Uo Units', 'Ligand Efficiency BEI', 'Ligand Efficiency LE', 'Ligand Efficiency LLE', 'Ligand Efficiency SEI', 'Potent

In [11]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, DataStructs
from rdkit.Chem import Descriptors
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Create directories for saving fingerprints
import os
os.makedirs('data/fingerprints', exist_ok=True)
os.makedirs('data/screening_fingerprints', exist_ok=True)
os.makedirs('models', exist_ok=True)

def generate_morgan_fingerprints(smiles_list, radius=2, nbits=2048, name="dataset"):
    """
    Generate Morgan fingerprints (ECFP4 equivalent)
    radius=2 gives ECFP4 (2 bonds diameter = 4 atoms)
    """
    fingerprints = []
    valid_indices = []
    valid_smiles = []
    
    print(f"Generating Morgan fingerprints (ECFP4) for {name}...")
    
    for idx, smiles in enumerate(smiles_list):
        if idx % 10000 == 0 and idx > 0:
            print(f"  Processed {idx}/{len(smiles_list)} compounds")
        
        try:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is not None:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nbits)
                # Convert to numpy array
                arr = np.zeros((nbits,), dtype=np.int8)
                DataStructs.ConvertToNumpyArray(fp, arr)
                fingerprints.append(arr)
                valid_indices.append(idx)
                valid_smiles.append(smiles)
        except Exception as e:
            continue
    
    fingerprints_array = np.array(fingerprints)
    print(f"  Generated {len(fingerprints)} fingerprints from {len(smiles_list)} SMILES")
    print(f"  Success rate: {len(fingerprints)/len(smiles_list)*100:.2f}%")
    
    return fingerprints_array, valid_indices, valid_smiles

def generate_maccs_fingerprints(smiles_list, name="dataset"):
    """
    Generate MACCS fingerprints (166 structural keys)
    """
    fingerprints = []
    valid_indices = []
    valid_smiles = []
    
    print(f"Generating MACCS fingerprints for {name}...")
    
    for idx, smiles in enumerate(smiles_list):
        if idx % 10000 == 0 and idx > 0:
            print(f"  Processed {idx}/{len(smiles_list)} compounds")
        
        try:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is not None:
                fp = MACCSkeys.GenMACCSKeys(mol)
                # Convert to numpy array
                arr = np.zeros((167,), dtype=np.int8)  # MACCS returns 167 bits
                DataStructs.ConvertToNumpyArray(fp, arr)
                fingerprints.append(arr)
                valid_indices.append(idx)
                valid_smiles.append(smiles)
        except Exception as e:
            continue
    
    fingerprints_array = np.array(fingerprints)
    print(f"  Generated {len(fingerprints)} fingerprints from {len(smiles_list)} SMILES")
    print(f"  Success rate: {len(fingerprints)/len(smiles_list)*100:.2f}%")
    
    return fingerprints_array, valid_indices, valid_smiles

# ============================================================================
# 1. PROCESS TRAINING DATA (CHEMBL compounds)
# ============================================================================
print("=" * 70)
print("STEP 1: Processing Training Data (CHEMBL LDHA compounds)")
print("=" * 70)

# Load your processed ChEMBL data
df_train = pd.read_csv('data/processed/chembl_ldha_processed.csv')

# Extract only active and inactive compounds (exclude intermediate if any)
df_train_clean = df_train[df_train['Activity_Label'].isin(['Active', 'Inactive'])].copy()
print(f"\nTraining compounds: {len(df_train_clean)}")
print(f"  - Active: {(df_train_clean['Activity_Binary']==1).sum()}")
print(f"  - Inactive: {(df_train_clean['Activity_Binary']==0).sum()}")

# Get SMILES and labels
train_smiles = df_train_clean['Smiles'].tolist()
train_labels = df_train_clean['Activity_Binary'].values
train_chembl_ids = df_train_clean['Molecule ChEMBL ID'].tolist()

# Generate Morgan fingerprints (ECFP4)
print("\n" + "-" * 50)
train_morgan_fps, train_morgan_idx, train_morgan_smiles = generate_morgan_fingerprints(
    train_smiles, radius=2, nbits=2048, name="Training Data"
)

# Generate MACCS fingerprints
print("\n" + "-" * 50)
train_maccs_fps, train_maccs_idx, train_maccs_smiles = generate_maccs_fingerprints(
    train_smiles, name="Training Data"
)

# Align labels with valid fingerprints
train_labels_morgan = train_labels[train_morgan_idx]
train_labels_maccs = train_labels[train_maccs_idx]

# Get corresponding metadata
train_metadata_morgan = df_train_clean.iloc[train_morgan_idx][['Molecule ChEMBL ID', 'Smiles', 'Activity_Label', 'pIC50']]
train_metadata_maccs = df_train_clean.iloc[train_maccs_idx][['Molecule ChEMBL ID', 'Smiles', 'Activity_Label', 'pIC50']]

# Save training fingerprints
print("\n" + "-" * 50)
print("Saving training fingerprints...")

# Morgan fingerprints
np.save('data/fingerprints/train_morgan_fps.npy', train_morgan_fps)
np.save('data/fingerprints/train_morgan_labels.npy', train_labels_morgan)
train_metadata_morgan.to_csv('data/fingerprints/train_morgan_metadata.csv', index=False)

# MACCS fingerprints
np.save('data/fingerprints/train_maccs_fps.npy', train_maccs_fps)
np.save('data/fingerprints/train_maccs_labels.npy', train_labels_maccs)
train_metadata_maccs.to_csv('data/fingerprints/train_maccs_metadata.csv', index=False)

print(f"✓ Morgan fingerprints saved: {train_morgan_fps.shape}")
print(f"✓ MACCS fingerprints saved: {train_maccs_fps.shape}")

# ============================================================================
# 2. PROCESS SCREENING DATA (Random compounds - 100,000)
# ============================================================================
print("\n" + "=" * 70)
print("STEP 2: Processing Screening Data (100,000 compounds)")
print("=" * 70)

# Load random screening compounds
df_screening = pd.read_csv('random.csv')
print(f"\nScreening compounds: {len(df_screening)}")
print(f"Columns: {df_screening.columns.tolist()}")

# Get SMILES
screening_smiles = df_screening['SMILES'].tolist()
screening_zincids = df_screening['zincid'].tolist()

# Generate Morgan fingerprints (ECFP4) for screening
print("\n" + "-" * 50)
screening_morgan_fps, screening_morgan_idx, screening_morgan_smiles = generate_morgan_fingerprints(
    screening_smiles, radius=2, nbits=2048, name="Screening Data"
)

# Generate MACCS fingerprints for screening
print("\n" + "-" * 50)
screening_maccs_fps, screening_maccs_idx, screening_maccs_smiles = generate_maccs_fingerprints(
    screening_smiles, name="Screening Data"
)

# Get corresponding metadata
screening_metadata_morgan = df_screening.iloc[screening_morgan_idx][['zincid', 'SMILES']]
screening_metadata_maccs = df_screening.iloc[screening_maccs_idx][['zincid', 'SMILES']]

# Save screening fingerprints
print("\n" + "-" * 50)
print("Saving screening fingerprints...")

# Morgan fingerprints
np.save('data/screening_fingerprints/screening_morgan_fps.npy', screening_morgan_fps)
screening_metadata_morgan.to_csv('data/screening_fingerprints/screening_morgan_metadata.csv', index=False)

# MACCS fingerprints
np.save('data/screening_fingerprints/screening_maccs_fps.npy', screening_maccs_fps)
screening_metadata_maccs.to_csv('data/screening_fingerprints/screening_maccs_metadata.csv', index=False)

print(f"✓ Screening Morgan fingerprints saved: {screening_morgan_fps.shape}")
print(f"✓ Screening MACCS fingerprints saved: {screening_maccs_fps.shape}")

# ============================================================================
# 3. SAVE SUMMARY REPORT
# ============================================================================
print("\n" + "=" * 70)
print("SUMMARY REPORT")
print("=" * 70)

summary = {
    'Training_Data': {
        'Total_Compounds': len(df_train_clean),
        'Active': int((train_labels == 1).sum()),
        'Inactive': int((train_labels == 0).sum()),
        'Morgan_FP_Shape': train_morgan_fps.shape,
        'Morgan_Valid_Compounds': len(train_morgan_fps),
        'MACCS_FP_Shape': train_maccs_fps.shape,
        'MACCS_Valid_Compounds': len(train_maccs_fps)
    },
    'Screening_Data': {
        'Total_Compounds': len(df_screening),
        'Morgan_FP_Shape': screening_morgan_fps.shape,
        'Morgan_Valid_Compounds': len(screening_morgan_fps),
        'Morgan_Success_Rate': f"{len(screening_morgan_fps)/len(df_screening)*100:.2f}%",
        'MACCS_FP_Shape': screening_maccs_fps.shape,
        'MACCS_Valid_Compounds': len(screening_maccs_fps),
        'MACCS_Success_Rate': f"{len(screening_maccs_fps)/len(df_screening)*100:.2f}%"
    }
}

summary_df = pd.DataFrame(summary).T
print("\n", summary_df)

# Save summary
summary_df.to_csv('data/fingerprints/fingerprint_generation_summary.csv')

# Save configuration for reproducibility
config = {
    'morgan_radius': 2,
    'morgan_nbits': 2048,
    'morgan_notes': 'ECFP4 equivalent (radius=2, 2 bonds diameter)',
    'maccs_version': 'RDKit MACCS166',
    'maccs_bits': 167,
    'date_processed': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'rdkit_version': Chem.rdBase.rdkitVersion
}

with open('data/fingerprints/fingerprint_config.pkl', 'wb') as f:
    pickle.dump(config, f)

print("\n✓ Configuration saved for reproducibility")
print("\nFiles saved:")
print("  Training Morgan: data/fingerprints/train_morgan_fps.npy")
print("  Training MACCS: data/fingerprints/train_maccs_fps.npy")
print("  Screening Morgan: data/screening_fingerprints/screening_morgan_fps.npy")
print("  Screening MACCS: data/screening_fingerprints/screening_maccs_fps.npy")
print("\n✅ Fingerprint generation complete!")

STEP 1: Processing Training Data (CHEMBL LDHA compounds)

Training compounds: 578
  - Active: 291
  - Inactive: 287

--------------------------------------------------
Generating Morgan fingerprints (ECFP4) for Training Data...
  Generated 578 fingerprints from 578 SMILES
  Success rate: 100.00%

--------------------------------------------------
Generating MACCS fingerprints for Training Data...
  Generated 578 fingerprints from 578 SMILES
  Success rate: 100.00%

--------------------------------------------------
Saving training fingerprints...
✓ Morgan fingerprints saved: (578, 2048)
✓ MACCS fingerprints saved: (578, 167)

STEP 2: Processing Screening Data (100,000 compounds)

Screening compounds: 100000
Columns: ['zincid', 'SMILES']

--------------------------------------------------
Generating Morgan fingerprints (ECFP4) for Screening Data...
  Processed 10000/100000 compounds
  Processed 20000/100000 compounds
  Processed 30000/100000 compounds
  Processed 40000/100000 compounds


In [12]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, matthews_corrcoef, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Create directories for outputs
import os
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def calculate_metrics(y_true, y_pred, y_pred_proba):
    """Calculate all evaluation metrics"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1_Score': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC_ROC': roc_auc_score(y_true, y_pred_proba),
        'True_Positives': tp,
        'True_Negatives': tn,
        'False_Positives': fp,
        'False_Negatives': fn,
        'False_Positive_Rate': fp / (fp + tn) if (fp + tn) > 0 else 0,
        'True_Negative_Rate': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'Sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0
    }
    return metrics

def plot_roc_curves(y_test, predictions, model_names, fingerprint_type, save_path):
    """Plot ROC curves for multiple models"""
    plt.figure(figsize=(10, 8))
    
    for name, pred in predictions.items():
        fpr, tpr, _ = roc_curve(y_test, pred)
        auc = roc_auc_score(y_test, pred)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)
    
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curves - {fingerprint_type} Fingerprints', fontsize=14)
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{save_path}/roc_curves_{fingerprint_type}.png', dpi=300, bbox_inches='tight')
    plt.close()

def plot_precision_recall_curves(y_test, predictions, model_names, fingerprint_type, save_path):
    """Plot Precision-Recall curves"""
    plt.figure(figsize=(10, 8))
    
    for name, pred in predictions.items():
        precision, recall, _ = precision_recall_curve(y_test, pred)
        auc_pr = np.trapz(precision, recall)
        plt.plot(recall, precision, label=f'{name} (AUPR = {auc_pr:.3f})', linewidth=2)
    
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(f'Precision-Recall Curves - {fingerprint_type} Fingerprints', fontsize=14)
    plt.legend(loc='lower left', fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{save_path}/pr_curves_{fingerprint_type}.png', dpi=300, bbox_inches='tight')
    plt.close()

def plot_confusion_matrices(y_test, y_preds, model_names, fingerprint_type, save_path):
    """Plot confusion matrices"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, (name, y_pred) in enumerate(y_preds.items()):
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                    xticklabels=['Inactive', 'Active'],
                    yticklabels=['Inactive', 'Active'])
        axes[idx].set_title(f'{name}', fontsize=12)
        axes[idx].set_xlabel('Predicted', fontsize=10)
        axes[idx].set_ylabel('Actual', fontsize=10)
    
    plt.suptitle(f'Confusion Matrices - {fingerprint_type} Fingerprints', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{save_path}/confusion_matrices_{fingerprint_type}.png', dpi=300, bbox_inches='tight')
    plt.close()

def create_dnn_model(input_dim, dropout_rate=0.3):
    """Create a Deep Neural Network model"""
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate/2),
        
        layers.Dense(64, activation='relu'),
        layers.Dropout(dropout_rate/2),
        
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================

def train_evaluate_models(fingerprint_name, X_train, X_test, y_train, y_test, feature_names):
    """Train and evaluate all three models on a given fingerprint type"""
    
    print(f"\n{'='*70}")
    print(f"TRAINING MODELS ON {fingerprint_name.upper()} FINGERPRINTS")
    print(f"{'='*70}")
    print(f"Training set: {X_train.shape}")
    print(f"Test set: {X_test.shape}")
    print(f"Active in test: {sum(y_test)}/{len(y_test)} ({sum(y_test)/len(y_test)*100:.1f}%)")
    
    # Standardize features for DNN and XGBoost
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    results = {}
    predictions = {}
    y_preds = {}
    models_saved = {}
    
    # ========================================================================
    # 1. RANDOM FOREST with Hyperparameter Tuning
    # ========================================================================
    print("\n" + "-"*50)
    print("1. Training Random Forest...")
    
    rf_param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'class_weight': ['balanced', None]
    }
    
    rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
    rf_grid = GridSearchCV(rf_base, rf_param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0)
    rf_grid.fit(X_train, y_train)
    
    rf_best = rf_grid.best_estimator_
    print(f"  Best parameters: {rf_grid.best_params_}")
    print(f"  Best CV AUC: {rf_grid.best_score_:.4f}")
    
    # 10-fold cross-validation
    rf_cv_scores = cross_val_score(rf_best, X_train, y_train, cv=10, scoring='roc_auc')
    print(f"  10-fold CV AUC: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std():.4f})")
    
    # Predictions
    rf_y_pred = rf_best.predict(X_test)
    rf_y_pred_proba = rf_best.predict_proba(X_test)[:, 1]
    
    rf_metrics = calculate_metrics(y_test, rf_y_pred, rf_y_pred_proba)
    rf_metrics['Model'] = 'Random Forest'
    rf_metrics['Fingerprint'] = fingerprint_name
    rf_metrics['Best_Params'] = str(rf_grid.best_params_)
    rf_metrics['CV_Mean_AUC'] = rf_cv_scores.mean()
    rf_metrics['CV_Std_AUC'] = rf_cv_scores.std()
    results['Random Forest'] = rf_metrics
    predictions['Random Forest'] = rf_y_pred_proba
    y_preds['Random Forest'] = rf_y_pred
    models_saved['Random Forest'] = rf_best
    
    # ========================================================================
    # 2. XGBOOST with Hyperparameter Tuning
    # ========================================================================
    print("\n" + "-"*50)
    print("2. Training XGBoost...")
    
    xgb_param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0]
    }
    
    xgb_base = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
    xgb_grid = GridSearchCV(xgb_base, xgb_param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0)
    xgb_grid.fit(X_train_scaled, y_train)
    
    xgb_best = xgb_grid.best_estimator_
    print(f"  Best parameters: {xgb_grid.best_params_}")
    print(f"  Best CV AUC: {xgb_grid.best_score_:.4f}")
    
    # 10-fold cross-validation
    xgb_cv_scores = cross_val_score(xgb_best, X_train_scaled, y_train, cv=10, scoring='roc_auc')
    print(f"  10-fold CV AUC: {xgb_cv_scores.mean():.4f} (+/- {xgb_cv_scores.std():.4f})")
    
    # Predictions
    xgb_y_pred = xgb_best.predict(X_test_scaled)
    xgb_y_pred_proba = xgb_best.predict_proba(X_test_scaled)[:, 1]
    
    xgb_metrics = calculate_metrics(y_test, xgb_y_pred, xgb_y_pred_proba)
    xgb_metrics['Model'] = 'XGBoost'
    xgb_metrics['Fingerprint'] = fingerprint_name
    xgb_metrics['Best_Params'] = str(xgb_grid.best_params_)
    xgb_metrics['CV_Mean_AUC'] = xgb_cv_scores.mean()
    xgb_metrics['CV_Std_AUC'] = xgb_cv_scores.std()
    results['XGBoost'] = xgb_metrics
    predictions['XGBoost'] = xgb_y_pred_proba
    y_preds['XGBoost'] = xgb_y_pred
    models_saved['XGBoost'] = xgb_best
    
    # ========================================================================
    # 3. DEEP NEURAL NETWORK
    # ========================================================================
    print("\n" + "-"*50)
    print("3. Training Deep Neural Network...")
    
    # Early stopping callback
    early_stop = callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=15,
        restore_best_weights=True,
        verbose=0
    )
    
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=0
    )
    
    # Create and train DNN
    dnn_model = create_dnn_model(X_train_scaled.shape[1])
    
    history = dnn_model.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        epochs=100,
        batch_size=32,
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )
    
    # 10-fold cross-validation for DNN (simplified due to computation time)
    dnn_cv_scores = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, val_idx in skf.split(X_train_scaled, y_train):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model_cv = create_dnn_model(X_train_scaled.shape[1])
        model_cv.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                     epochs=50, batch_size=32, verbose=0, callbacks=[early_stop])
        y_val_pred = model_cv.predict(X_val, verbose=0)
        dnn_cv_scores.append(roc_auc_score(y_val, y_val_pred))
    
    print(f"  5-fold CV AUC: {np.mean(dnn_cv_scores):.4f} (+/- {np.std(dnn_cv_scores):.4f})")
    
    # Predictions
    dnn_y_pred_proba = dnn_model.predict(X_test_scaled, verbose=0).flatten()
    dnn_y_pred = (dnn_y_pred_proba >= 0.5).astype(int)
    
    dnn_metrics = calculate_metrics(y_test, dnn_y_pred, dnn_y_pred_proba)
    dnn_metrics['Model'] = 'DNN'
    dnn_metrics['Fingerprint'] = fingerprint_name
    dnn_metrics['Best_Params'] = f'Layers: 512-256-128-64, Dropout: 0.3'
    dnn_metrics['CV_Mean_AUC'] = np.mean(dnn_cv_scores)
    dnn_metrics['CV_Std_AUC'] = np.std(dnn_cv_scores)
    results['DNN'] = dnn_metrics
    predictions['DNN'] = dnn_y_pred_proba
    y_preds['DNN'] = dnn_y_pred
    models_saved['DNN'] = dnn_model
    
    # Save scaler
    models_saved['Scaler'] = scaler
    
    # ========================================================================
    # SAVE RESULTS
    # ========================================================================
    
    # Plot curves
    plot_roc_curves(y_test, predictions, list(predictions.keys()), fingerprint_name, 'results/figures')
    plot_precision_recall_curves(y_test, predictions, list(predictions.keys()), fingerprint_name, 'results/figures')
    plot_confusion_matrices(y_test, y_preds, list(y_preds.keys()), fingerprint_name, 'results/figures')
    
    # Save models
    for name, model in models_saved.items():
        with open(f'models/{fingerprint_name}_{name.replace(" ", "_")}.pkl', 'wb') as f:
            pickle.dump(model, f)
    
    # Save training history plot for DNN
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['auc'], label='Training AUC')
    plt.plot(history.history['val_auc'], label='Validation AUC')
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('AUC', fontsize=12)
    plt.title(f'DNN Training History - {fingerprint_name} Fingerprints', fontsize=14)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(f'results/figures/dnn_training_{fingerprint_name}.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    return results

# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("="*70)
print("MODEL TRAINING PIPELINE FOR LDHA INHIBITOR PREDICTION")
print("="*70)

# Load fingerprints
print("\nLoading fingerprints...")

# Morgan fingerprints
X_morgan = np.load('data/fingerprints/train_morgan_fps.npy')
y = np.load('data/fingerprints/train_morgan_labels.npy')

# MACCS fingerprints
X_maccs = np.load('data/fingerprints/train_maccs_fps.npy')

print(f"Morgan fingerprints shape: {X_morgan.shape}")
print(f"MACCS fingerprints shape: {X_maccs.shape}")
print(f"Labels shape: {y.shape}")
print(f"Active: {sum(y)}, Inactive: {len(y)-sum(y)}")

# Split data (80/20)
X_morgan_train, X_morgan_test, y_train, y_test = train_test_split(
    X_morgan, y, test_size=0.2, random_state=42, stratify=y
)

X_maccs_train, X_maccs_test, _, _ = train_test_split(
    X_maccs, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain/Test split: {len(X_morgan_train)}/{len(X_morgan_test)}")

# Train models on Morgan fingerprints
results_morgan = train_evaluate_models(
    'Morgan', X_morgan_train, X_morgan_test, y_train, y_test, 'Morgan_FP'
)

# Train models on MACCS fingerprints
results_maccs = train_evaluate_models(
    'MACCS', X_maccs_train, X_maccs_test, y_train, y_test, 'MACCS_FP'
)

# ============================================================================
# COMPILE AND SAVE ALL RESULTS
# ============================================================================

# Combine results
all_results = []
for model_name, metrics in results_morgan.items():
    all_results.append(metrics)
for model_name, metrics in results_maccs.items():
    all_results.append(metrics)

results_df = pd.DataFrame(all_results)

# Reorder columns for better readability
column_order = ['Model', 'Fingerprint', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 
                'MCC', 'AUC_ROC', 'Specificity', 'Sensitivity', 'False_Positive_Rate',
                'True_Negative_Rate', 'True_Positives', 'True_Negatives', 
                'False_Positives', 'False_Negatives', 'CV_Mean_AUC', 'CV_Std_AUC', 
                'Best_Params']
results_df = results_df[column_order]

# Save results
results_df.to_csv('results/model_performance_summary.csv', index=False)

# Print summary
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)
print("\n", results_df.to_string(index=False))

# Create a comparative bar plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'AUC_ROC']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx//2, idx%2]
    
    morgan_vals = results_df[results_df['Fingerprint']=='Morgan'][metric].values
    maccs_vals = results_df[results_df['Fingerprint']=='MACCS'][metric].values
    models = results_df[results_df['Fingerprint']=='Morgan']['Model'].values
    
    x = np.arange(len(models))
    width = 0.35
    
    ax.bar(x - width/2, morgan_vals, width, label='Morgan', color='royalblue')
    ax.bar(x + width/2, maccs_vals, width, label='MACCS', color='coral')
    
    ax.set_xlabel('Model', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric} Comparison', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend()
    ax.grid(alpha=0.3, axis='y')

plt.suptitle('Model Performance Comparison: Morgan vs MACCS Fingerprints', fontsize=14)
plt.tight_layout()
plt.savefig('results/figures/performance_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n" + "="*70)
print("SAVED FILES:")
print("="*70)
print("✓ results/model_performance_summary.csv - Complete metrics table")
print("✓ results/figures/roc_curves_Morgan.png - ROC curves for Morgan")
print("✓ results/figures/roc_curves_MACCS.png - ROC curves for MACCS")
print("✓ results/figures/pr_curves_Morgan.png - Precision-Recall curves for Morgan")
print("✓ results/figures/pr_curves_MACCS.png - Precision-Recall curves for MACCS")
print("✓ results/figures/confusion_matrices_Morgan.png - Confusion matrices for Morgan")
print("✓ results/figures/confusion_matrices_MACCS.png - Confusion matrices for MACCS")
print("✓ results/figures/performance_comparison.png - Bar plot comparison")
print("✓ results/figures/dnn_training_Morgan.png - DNN training history")
print("✓ results/figures/dnn_training_MACCS.png - DNN training history")
print("\n✓ MODELS SAVED:")
print("  - models/Morgan_Random_Forest.pkl")
print("  - models/Morgan_XGBoost.pkl")
print("  - models/Morgan_DNN.pkl")
print("  - models/Morgan_Scaler.pkl")
print("  - models/MACCS_Random_Forest.pkl")
print("  - models/MACCS_XGBoost.pkl")
print("  - models/MACCS_DNN.pkl")
print("  - models/MACCS_Scaler.pkl")

print("\n✅ MODEL TRAINING COMPLETE!")

MODEL TRAINING PIPELINE FOR LDHA INHIBITOR PREDICTION

Loading fingerprints...
Morgan fingerprints shape: (578, 2048)
MACCS fingerprints shape: (578, 167)
Labels shape: (578,)
Active: 291, Inactive: 287

Train/Test split: 462/116

TRAINING MODELS ON MORGAN FINGERPRINTS
Training set: (462, 2048)
Test set: (116, 2048)
Active in test: 58/116 (50.0%)

--------------------------------------------------
1. Training Random Forest...
  Best parameters: {'class_weight': None, 'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
  Best CV AUC: 0.9858
  10-fold CV AUC: 0.9819 (+/- 0.0193)

--------------------------------------------------
2. Training XGBoost...
  Best parameters: {'colsample_bytree': 0.9, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'subsample': 0.8}
  Best CV AUC: 0.9800
  10-fold CV AUC: 0.9800 (+/- 0.0230)

--------------------------------------------------
3. Training Deep Neural Network...
  5-fold CV AUC: 0.9545 (+/- 0.0285)

T

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# Set publication-ready style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1

# Create directory for publication figures
import os
os.makedirs('results/publication_figures', exist_ok=True)

# Load results
results_df = pd.read_csv('results/model_performance_summary.csv')

# ============================================================================
# FIGURE 1: Heatmap of Performance Metrics
# ============================================================================
print("Generating Figure 1: Performance Heatmap...")

# Prepare data for heatmap
metrics_for_heatmap = ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'MCC', 'AUC_ROC', 'Specificity']
heatmap_data = results_df[['Model', 'Fingerprint'] + metrics_for_heatmap].copy()
heatmap_data['Model_FP'] = heatmap_data['Model'] + ' (' + heatmap_data['Fingerprint'] + ')'

# Create pivot table
heatmap_pivot = heatmap_data.pivot_table(
    values=metrics_for_heatmap, 
    index='Model_FP', 
    aggfunc='first'
)

# Create heatmap
fig, ax = plt.subplots(figsize=(12, 8))

# Create heatmap with custom colormap
im = ax.imshow(heatmap_pivot.values, cmap='RdYlGn', aspect='auto', vmin=0.7, vmax=1.0)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Performance Score', fontsize=12, fontweight='bold')

# Set ticks and labels
ax.set_xticks(np.arange(len(metrics_for_heatmap)))
ax.set_yticks(np.arange(len(heatmap_pivot.index)))
ax.set_xticklabels(metrics_for_heatmap, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(heatmap_pivot.index, fontsize=10)

# Add text annotations
for i in range(len(heatmap_pivot.index)):
    for j in range(len(metrics_for_heatmap)):
        value = heatmap_pivot.values[i, j]
        text_color = 'white' if value < 0.85 else 'black'
        ax.text(j, i, f'{value:.3f}', ha='center', va='center', 
                color=text_color, fontsize=9, fontweight='bold')

# Add dividing lines
ax.axhline(y=2.5, color='black', linewidth=2, linestyle='-')
ax.axhline(y=5.5, color='black', linewidth=2, linestyle='-')

# Add group labels
ax.text(-0.5, 1, 'Morgan\nFPs', ha='center', va='center', 
        transform=ax.transData, fontsize=10, fontweight='bold', rotation=90)
ax.text(-0.5, 4, 'MACCS\nFPs', ha='center', va='center', 
        transform=ax.transData, fontsize=10, fontweight='bold', rotation=90)

ax.set_title('Model Performance Comparison: Morgan vs MACCS Fingerprints', 
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('results/publication_figures/Figure1_Performance_Heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('results/publication_figures/Figure1_Performance_Heatmap.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Figure 1 saved")

# ============================================================================
# FIGURE 2: Bar Plot with Error Bars (CV Results)
# ============================================================================
print("Generating Figure 2: CV Performance Bar Plot...")

fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(results_df['Model'].unique()))
width = 0.35
models = results_df['Model'].unique()

# Prepare data
morgan_cv = results_df[results_df['Fingerprint']=='Morgan']['CV_Mean_AUC'].values
morgan_std = results_df[results_df['Fingerprint']=='Morgan']['CV_Std_AUC'].values
maccs_cv = results_df[results_df['Fingerprint']=='MACCS']['CV_Mean_AUC'].values
maccs_std = results_df[results_df['Fingerprint']=='MACCS']['CV_Std_AUC'].values

# Create bars
bars1 = ax.bar(x - width/2, morgan_cv, width, yerr=morgan_std, 
               label='Morgan Fingerprints', color='#2E86AB', 
               capsize=5, error_kw={'linewidth': 1.5, 'elinewidth': 1.5})
bars2 = ax.bar(x + width/2, maccs_cv, width, yerr=maccs_std,
               label='MACCS Fingerprints', color='#A23B72', 
               capsize=5, error_kw={'linewidth': 1.5, 'elinewidth': 1.5})

# Customize plot
ax.set_ylabel('10-fold Cross-Validation AUC', fontsize=12, fontweight='bold')
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_title('Cross-Validation Performance with Standard Deviation', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0.85, 1.0)
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.003,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('results/publication_figures/Figure2_CV_Performance.png', dpi=300, bbox_inches='tight')
plt.savefig('results/publication_figures/Figure2_CV_Performance.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Figure 2 saved")

# ============================================================================
# FIGURE 3: Radar Chart for Multi-Metric Comparison
# ============================================================================
print("Generating Figure 3: Radar Chart...")

from math import pi

metrics_radar = ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'MCC', 'AUC_ROC']
colors = ['#2E86AB', '#A23B72', '#F18F01', '#048A81', '#C73E1D', '#6A4C93']

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

# Number of variables
N = len(metrics_radar)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Plot each model
for idx, (model_name, fingerprint) in enumerate(zip(results_df['Model'], results_df['Fingerprint'])):
    values = results_df.loc[idx, metrics_radar].values.tolist()
    values += values[:1]  # Close the loop
    
    ax.plot(angles, values, 'o-', linewidth=2, 
            label=f'{model_name} ({fingerprint})', color=colors[idx % len(colors)])
    ax.fill(angles, values, alpha=0.1, color=colors[idx % len(colors)])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_radar, fontsize=11, fontweight='bold')
ax.set_ylim(0.7, 1.0)
ax.set_yticks([0.7, 0.8, 0.9, 1.0])
ax.set_yticklabels(['0.7', '0.8', '0.9', '1.0'], fontsize=9)
ax.set_title('Multi-Metric Performance Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=9)

plt.tight_layout()
plt.savefig('results/publication_figures/Figure3_Radar_Chart.png', dpi=300, bbox_inches='tight')
plt.savefig('results/publication_figures/Figure3_Radar_Chart.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Figure 3 saved")

# ============================================================================
# FIGURE 4: Confusion Matrix Summary
# ============================================================================
print("Generating Figure 4: Confusion Matrix Summary...")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (model_name, fingerprint) in enumerate(zip(results_df['Model'], results_df['Fingerprint'])):
    tn = results_df.loc[idx, 'True_Negatives']
    fp = results_df.loc[idx, 'False_Positives']
    fn = results_df.loc[idx, 'False_Negatives']
    tp = results_df.loc[idx, 'True_Positives']
    
    cm = np.array([[tn, fp], [fn, tp]])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Inactive', 'Active'],
                yticklabels=['Inactive', 'Active'],
                annot_kws={'size': 14, 'weight': 'bold'})
    
    axes[idx].set_title(f'{model_name} - {fingerprint}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=11)
    axes[idx].set_ylabel('Actual', fontsize=11)
    
    # Add metrics text
    acc = results_df.loc[idx, 'Accuracy']
    auc = results_df.loc[idx, 'AUC_ROC']
    axes[idx].text(0.5, -0.15, f'Acc: {acc:.3f} | AUC: {auc:.3f}', 
                   transform=axes[idx].transAxes, ha='center', fontsize=10)

plt.suptitle('Confusion Matrices for All Models and Fingerprint Types', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/publication_figures/Figure4_Confusion_Matrices.png', dpi=300, bbox_inches='tight')
plt.savefig('results/publication_figures/Figure4_Confusion_Matrices.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Figure 4 saved")

# ============================================================================
# FIGURE 5: Feature Importance (Top 20) - For Best Model
# ============================================================================
print("Generating Figure 5: Feature Importance...")

# Load best model (Random Forest with Morgan had highest AUC)
import pickle
best_model = pickle.load(open('models/Morgan_Random_Forest.pkl', 'rb'))
feature_importance = best_model.feature_importances_
top_features_idx = np.argsort(feature_importance)[-20:]

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = np.arange(len(top_features_idx))

ax.barh(y_pos, feature_importance[top_features_idx], color='#2E86AB', edgecolor='black', alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([f'Bit {i}' for i in top_features_idx], fontsize=9)
ax.set_xlabel('Feature Importance', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Most Important Morgan Fingerprint Features\n(Random Forest Model)', 
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('results/publication_figures/Figure5_Feature_Importance.png', dpi=300, bbox_inches='tight')
plt.savefig('results/publication_figures/Figure5_Feature_Importance.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Figure 5 saved")

# ============================================================================
# FIGURE 6: Performance Summary Table (Publication Ready)
# ============================================================================
print("Generating Figure 6: Performance Summary Table...")

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('tight')
ax.axis('off')

# Prepare table data
table_data = results_df[['Model', 'Fingerprint', 'Accuracy', 'Precision', 
                         'Recall', 'F1_Score', 'MCC', 'AUC_ROC', 'CV_Mean_AUC']].copy()
table_data = table_data.round(4)
table_data['CV_Mean_AUC'] = table_data['CV_Mean_AUC'].round(4)

# Create table
table = ax.table(cellText=table_data.values,
                 colLabels=table_data.columns,
                 cellLoc='center',
                 loc='center',
                 colWidths=[0.08, 0.08, 0.08, 0.08, 0.08, 0.08, 0.08, 0.08, 0.12])

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)

# Color code the best values
for i in range(len(table_data)):
    for j, col in enumerate(table_data.columns):
        if col in ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'MCC', 'AUC_ROC', 'CV_Mean_AUC']:
            cell_value = table_data.iloc[i, j]
            if cell_value == table_data[col].max():
                table[(i+1, j)].set_facecolor('#90EE90')
            elif cell_value == table_data[col].min():
                table[(i+1, j)].set_facecolor('#FFB6C1')

# Header styling
for j, col in enumerate(table_data.columns):
    table[(0, j)].set_facecolor('#40466e')
    table[(0, j)].set_text_props(weight='bold', color='white')

ax.set_title('Model Performance Summary Table\n(Best values highlighted in green)', 
             fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('results/publication_figures/Figure6_Performance_Table.png', dpi=300, bbox_inches='tight')
plt.savefig('results/publication_figures/Figure6_Performance_Table.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Figure 6 saved")

# ============================================================================
# FIGURE 7: Correlation Matrix of Metrics
# ============================================================================
print("Generating Figure 7: Metrics Correlation Matrix...")

metrics_corr = results_df[['Accuracy', 'Precision', 'Recall', 'F1_Score', 'MCC', 'AUC_ROC']]
corr_matrix = metrics_corr.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
heatmap = sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', 
                      cmap='coolwarm', center=0, square=True,
                      linewidths=1, cbar_kws={"shrink": 0.8},
                      annot_kws={'size': 11, 'weight': 'bold'})

ax.set_title('Correlation Matrix of Evaluation Metrics', 
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('results/publication_figures/Figure7_Metrics_Correlation.png', dpi=300, bbox_inches='tight')
plt.savefig('results/publication_figures/Figure7_Metrics_Correlation.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Figure 7 saved")

# ============================================================================
# FIGURE 8: Model Comparison Box Plot (If multiple runs were done)
# ============================================================================
print("Generating Figure 8: Model Comparison Summary...")

fig, ax = plt.subplots(figsize=(10, 6))

# Create grouped bar plot for all metrics
metrics_plot = ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'MCC', 'AUC_ROC']
x = np.arange(len(metrics_plot))
width = 0.15
multiplier = 0

models_to_plot = ['Random Forest', 'XGBoost', 'DNN']
fingerprints = ['Morgan', 'MACCS']

for model in models_to_plot:
    for fp in fingerprints:
        subset = results_df[(results_df['Model']==model) & (results_df['Fingerprint']==fp)]
        if len(subset) > 0:
            values = subset[metrics_plot].values.flatten()
            offset = width * multiplier
            ax.bar(x + offset, values, width, label=f'{model} ({fp})', alpha=0.8)
            multiplier += 1

ax.set_xlabel('Evaluation Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Comprehensive Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_plot, rotation=45, ha='right')
ax.set_ylim(0.7, 1.0)
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results/publication_figures/Figure8_Comprehensive_Comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('results/publication_figures/Figure8_Comprehensive_Comparison.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Figure 8 saved")

# ============================================================================
# SUMMARY REPORT
# ============================================================================
print("\n" + "="*70)
print("PUBLICATION FIGURES GENERATED SUCCESSFULLY!")
print("="*70)
print("\nFigures saved in 'results/publication_figures/':")
print("  📊 Figure1_Performance_Heatmap.png/pdf - Heatmap of all metrics")
print("  📊 Figure2_CV_Performance.png/pdf - CV AUC with error bars")
print("  📊 Figure3_Radar_Chart.png/pdf - Multi-metric radar chart")
print("  📊 Figure4_Confusion_Matrices.png/pdf - All confusion matrices")
print("  📊 Figure5_Feature_Importance.png/pdf - Top 20 features")
print("  📊 Figure6_Performance_Table.png/pdf - Summary table")
print("  📊 Figure7_Metrics_Correlation.png/pdf - Metrics correlation")
print("  📊 Figure8_Comprehensive_Comparison.png/pdf - Grouped bar plot")

# Create a figure caption file for the paper
caption_text = """
FIGURE CAPTIONS FOR MANUSCRIPT:

Figure 1. Performance heatmap comparing Morgan and MACCS fingerprints across six machine learning models. 
Values range from 0.7 to 1.0 (green = higher performance). Morgan fingerprints consistently outperform 
MACCS fingerprints across all evaluation metrics.

Figure 2. Cross-validation performance (10-fold for RF/XGBoost, 5-fold for DNN) with standard deviation bars. 
Random Forest with Morgan fingerprints achieved the highest CV AUC (0.982 ± 0.019).

Figure 3. Radar chart showing multi-metric performance comparison. The chart demonstrates the balanced 
performance of all models across Accuracy, Precision, Recall, F1 Score, MCC, and AUC-ROC.

Figure 4. Confusion matrices for all six model-fingerprint combinations. True positives, true negatives, 
false positives, and false negatives are shown for the 20% holdout test set (n=116 compounds).

Figure 5. Top 20 most important Morgan fingerprint features identified by the best-performing Random Forest 
model. Feature importance scores indicate bits that contribute most to LDHA inhibitor classification.

Figure 6. Summary table of all performance metrics. Best values for each metric are highlighted in green. 
The Random Forest model with Morgan fingerprints achieved the highest overall performance.

Figure 7. Correlation matrix of evaluation metrics showing strong positive correlations between related 
metrics (e.g., F1 Score with Accuracy and AUC-ROC).

Figure 8. Comprehensive comparison of all models across six evaluation metrics. Morgan fingerprints 
(especially with Random Forest) consistently outperform MACCS fingerprints.
"""

with open('results/publication_figures/FIGURE_CAPTIONS.txt', 'w') as f:
    f.write(caption_text)

print("\n📝 Figure captions saved: results/publication_figures/FIGURE_CAPTIONS.txt")
print("\n✅ All publication-ready figures generated!")

# Display summary of best model
print("\n" + "="*70)
print("BEST MODEL SUMMARY FOR PUBLICATION")
print("="*70)
best_row = results_df.loc[results_df['AUC_ROC'].idxmax()]
print(f"\n🏆 Best Model: {best_row['Model']} with {best_row['Fingerprint']} fingerprints")
print(f"   Accuracy:  {best_row['Accuracy']:.4f}")
print(f"   Precision: {best_row['Precision']:.4f}")
print(f"   Recall:    {best_row['Recall']:.4f}")
print(f"   F1 Score:  {best_row['F1_Score']:.4f}")
print(f"   MCC:       {best_row['MCC']:.4f}")
print(f"   AUC-ROC:   {best_row['AUC_ROC']:.4f}")
print(f"   10-fold CV AUC: {best_row['CV_Mean_AUC']:.4f} ± {best_row['CV_Std_AUC']:.4f}")

Generating Figure 1: Performance Heatmap...
  ✓ Figure 1 saved
Generating Figure 2: CV Performance Bar Plot...
  ✓ Figure 2 saved
Generating Figure 3: Radar Chart...
  ✓ Figure 3 saved
Generating Figure 4: Confusion Matrix Summary...
  ✓ Figure 4 saved
Generating Figure 5: Feature Importance...
  ✓ Figure 5 saved
Generating Figure 6: Performance Summary Table...
  ✓ Figure 6 saved
Generating Figure 7: Metrics Correlation Matrix...
  ✓ Figure 7 saved
Generating Figure 8: Model Comparison Summary...
  ✓ Figure 8 saved

PUBLICATION FIGURES GENERATED SUCCESSFULLY!

Figures saved in 'results/publication_figures/':
  📊 Figure1_Performance_Heatmap.png/pdf - Heatmap of all metrics
  📊 Figure2_CV_Performance.png/pdf - CV AUC with error bars
  📊 Figure3_Radar_Chart.png/pdf - Multi-metric radar chart
  📊 Figure4_Confusion_Matrices.png/pdf - All confusion matrices
  📊 Figure5_Feature_Importance.png/pdf - Top 20 features
  📊 Figure6_Performance_Table.png/pdf - Summary table
  📊 Figure7_Metrics_Corr

In [14]:
import pandas as pd
import numpy as np
import pickle
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, DataStructs
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
import os
from datetime import datetime
import tensorflow as tf

# Suppress TensorFlow warnings
tf.get_logger().setLevel('ERROR')

# Create directories for screening results
os.makedirs('screening_results', exist_ok=True)
os.makedirs('screening_results/consensus_hits', exist_ok=True)

# ============================================================================
# 1. LOAD ALL TRAINED MODELS
# ============================================================================
print("="*80)
print("VIRTUAL SCREENING PIPELINE FOR LDHA INHIBITORS")
print("="*80)
print("\nStep 1: Loading trained models...")

models = {}

# Load Morgan fingerprint models
models['Morgan_RF'] = pickle.load(open('models/Morgan_Random_Forest.pkl', 'rb'))
models['Morgan_XGB'] = pickle.load(open('models/Morgan_XGBoost.pkl', 'rb'))
models['Morgan_DNN'] = pickle.load(open('models/Morgan_DNN.pkl', 'rb'))
models['Morgan_Scaler'] = pickle.load(open('models/Morgan_Scaler.pkl', 'rb'))

# Load MACCS fingerprint models
models['MACCS_RF'] = pickle.load(open('models/MACCS_Random_Forest.pkl', 'rb'))
models['MACCS_XGB'] = pickle.load(open('models/MACCS_XGBoost.pkl', 'rb'))
models['MACCS_DNN'] = pickle.load(open('models/MACCS_DNN.pkl', 'rb'))
models['MACCS_Scaler'] = pickle.load(open('models/MACCS_Scaler.pkl', 'rb'))

print("✓ All 6 models loaded successfully")

# ============================================================================
# 2. LOAD SCREENING DATA AND FINGERPRINTS
# ============================================================================
print("\nStep 2: Loading screening data and fingerprints...")

# Load screening metadata
screening_metadata_morgan = pd.read_csv('data/screening_fingerprints/screening_morgan_metadata.csv')
screening_metadata_maccs = pd.read_csv('data/screening_fingerprints/screening_maccs_metadata.csv')

# Load fingerprints
screening_morgan_fps = np.load('data/screening_fingerprints/screening_morgan_fps.npy')
screening_maccs_fps = np.load('data/screening_fingerprints/screening_maccs_fps.npy')

print(f"✓ Screening compounds: {len(screening_metadata_morgan)}")
print(f"  Morgan fingerprints: {screening_morgan_fps.shape}")
print(f"  MACCS fingerprints: {screening_maccs_fps.shape}")

# ============================================================================
# 3. MAKE PREDICTIONS WITH ALL MODELS
# ============================================================================
print("\nStep 3: Making predictions with all models...")

# Scale Morgan fingerprints for DNN and XGBoost
print("  Scaling Morgan fingerprints...")
screening_morgan_scaled = models['Morgan_Scaler'].transform(screening_morgan_fps)

# Scale MACCS fingerprints
print("  Scaling MACCS fingerprints...")
screening_maccs_scaled = models['MACCS_Scaler'].transform(screening_maccs_fps)

# Initialize arrays for predictions
predictions = {}

# Morgan model predictions
print("\n  Predicting with Morgan models...")
predictions['Morgan_RF_proba'] = models['Morgan_RF'].predict_proba(screening_morgan_fps)[:, 1]
predictions['Morgan_XGB_proba'] = models['Morgan_XGB'].predict_proba(screening_morgan_scaled)[:, 1]
predictions['Morgan_DNN_proba'] = models['Morgan_DNN'].predict(screening_morgan_scaled, verbose=0).flatten()

# Morgan binary predictions (0.5 threshold)
predictions['Morgan_RF_pred'] = (predictions['Morgan_RF_proba'] >= 0.5).astype(int)
predictions['Morgan_XGB_pred'] = (predictions['Morgan_XGB_proba'] >= 0.5).astype(int)
predictions['Morgan_DNN_pred'] = (predictions['Morgan_DNN_proba'] >= 0.5).astype(int)

# MACCS model predictions
print("  Predicting with MACCS models...")
predictions['MACCS_RF_proba'] = models['MACCS_RF'].predict_proba(screening_maccs_fps)[:, 1]
predictions['MACCS_XGB_proba'] = models['MACCS_XGB'].predict_proba(screening_maccs_scaled)[:, 1]
predictions['MACCS_DNN_proba'] = models['MACCS_DNN'].predict(screening_maccs_scaled, verbose=0).flatten()

# MACCS binary predictions
predictions['MACCS_RF_pred'] = (predictions['MACCS_RF_proba'] >= 0.5).astype(int)
predictions['MACCS_XGB_pred'] = (predictions['MACCS_XGB_proba'] >= 0.5).astype(int)
predictions['MACCS_DNN_pred'] = (predictions['MACCS_DNN_proba'] >= 0.5).astype(int)

print("✓ Predictions completed for all 6 models")

# ============================================================================
# 4. CREATE COMPREHENSIVE RESULTS DATAFRAME
# ============================================================================
print("\nStep 4: Creating comprehensive results dataframe...")

# Use Morgan metadata as base (same compounds as MACCS)
results_df = screening_metadata_morgan.copy()

# Add all predictions
results_df['Morgan_RF_Prob'] = predictions['Morgan_RF_proba']
results_df['Morgan_XGB_Prob'] = predictions['Morgan_XGB_proba']
results_df['Morgan_DNN_Prob'] = predictions['Morgan_DNN_proba']
results_df['Morgan_RF_Pred'] = predictions['Morgan_RF_pred']
results_df['Morgan_XGB_Pred'] = predictions['Morgan_XGB_pred']
results_df['Morgan_DNN_Pred'] = predictions['Morgan_DNN_pred']

results_df['MACCS_RF_Prob'] = predictions['MACCS_RF_proba']
results_df['MACCS_XGB_Prob'] = predictions['MACCS_XGB_proba']
results_df['MACCS_DNN_Prob'] = predictions['MACCS_DNN_proba']
results_df['MACCS_RF_Pred'] = predictions['MACCS_RF_pred']
results_df['MACCS_XGB_Pred'] = predictions['MACCS_XGB_pred']
results_df['MACCS_DNN_Pred'] = predictions['MACCS_DNN_pred']

# Calculate consensus scores
# Consensus 1: Average probability across all 6 models
prob_columns = ['Morgan_RF_Prob', 'Morgan_XGB_Prob', 'Morgan_DNN_Prob',
                'MACCS_RF_Prob', 'MACCS_XGB_Prob', 'MACCS_DNN_Prob']
results_df['Consensus_Avg_Prob'] = results_df[prob_columns].mean(axis=1)

# Consensus 2: Weighted average (higher weight to better performing models)
# Based on test set performance: RF_Morgan (0.989), XGB_Morgan (0.984), DNN_Morgan (0.983)
# RF_MACCS (0.977), XGB_MACCS (0.979), DNN_MACCS (0.967)
weights = {
    'Morgan_RF_Prob': 0.20, 'Morgan_XGB_Prob': 0.19, 'Morgan_DNN_Prob': 0.18,
    'MACCS_RF_Prob': 0.15, 'MACCS_XGB_Prob': 0.16, 'MACCS_DNN_Prob': 0.12
}
results_df['Consensus_Weighted_Prob'] = sum(results_df[col] * weight for col, weight in weights.items())

# Consensus 3: Voting score (0-6, number of models predicting active)
vote_columns = ['Morgan_RF_Pred', 'Morgan_XGB_Pred', 'Morgan_DNN_Pred',
                'MACCS_RF_Pred', 'MACCS_XGB_Pred', 'MACCS_DNN_Pred']
results_df['Consensus_Vote_Count'] = results_df[vote_columns].sum(axis=1)

# Consensus 4: Strict consensus (all models agree)
results_df['Strict_Consensus'] = (results_df['Consensus_Vote_Count'] == 6).astype(int)

# Consensus 5: Majority consensus (>=4 models agree)
results_df['Majority_Consensus'] = (results_df['Consensus_Vote_Count'] >= 4).astype(int)

# ============================================================================
# 5. SAVE COMPLETE SCREENING RESULTS
# ============================================================================
print("\nStep 5: Saving screening results...")

# Save full results (compressed for large file)
results_df.to_csv('screening_results/full_screening_results.csv.gz', 
                  index=False, compression='gzip')
print(f"✓ Full results saved: screening_results/full_screening_results.csv.gz")
print(f"  File size: {os.path.getsize('screening_results/full_screening_results.csv.gz') / 1e6:.2f} MB")

# Save top 10% by consensus probability
top_10_percent = results_df.nlargest(int(len(results_df) * 0.1), 'Consensus_Avg_Prob')
top_10_percent.to_csv('screening_results/top_10_percent_hits.csv', index=False)
print(f"✓ Top 10% hits ({len(top_10_percent)} compounds) saved")

# ============================================================================
# 6. IDENTIFY CONSENSUS HITS (Active by all models)
# ============================================================================
print("\nStep 6: Identifying consensus hits...")

# Strategy 1: Strict consensus (all 6 models predict active)
strict_hits = results_df[results_df['Strict_Consensus'] == 1].copy()
print(f"\n  Strategy 1 - Strict Consensus (6/6 models):")
print(f"    Hits found: {len(strict_hits)} compounds")
print(f"    Percentage: {len(strict_hits)/len(results_df)*100:.3f}%")

# Strategy 2: Majority consensus (>=4 models predict active)
majority_hits = results_df[results_df['Majority_Consensus'] == 1].copy()
print(f"\n  Strategy 2 - Majority Consensus (≥4/6 models):")
print(f"    Hits found: {len(majority_hits)} compounds")
print(f"    Percentage: {len(majority_hits)/len(results_df)*100:.3f}%")

# Strategy 3: High confidence hits (Avg probability > 0.8 AND vote count >= 4)
high_confidence_hits = results_df[
    (results_df['Consensus_Avg_Prob'] > 0.8) & 
    (results_df['Consensus_Vote_Count'] >= 4)
].copy()
print(f"\n  Strategy 3 - High Confidence (Prob>0.8 & Vote≥4):")
print(f"    Hits found: {len(high_confidence_hits)} compounds")
print(f"    Percentage: {len(high_confidence_hits)/len(results_df)*100:.3f}%")

# Strategy 4: Very high confidence (Prob > 0.9 AND all models agree)
very_high_confidence = results_df[
    (results_df['Consensus_Avg_Prob'] > 0.9) & 
    (results_df['Strict_Consensus'] == 1)
].copy()
print(f"\n  Strategy 4 - Very High Confidence (Prob>0.9 & 6/6):")
print(f"    Hits found: {len(very_high_confidence)} compounds")
print(f"    Percentage: {len(very_high_confidence)/len(results_df)*100:.3f}%")

# ============================================================================
# 7. SAVE CONSENSUS HITS WITH STATISTICS
# ============================================================================
print("\nStep 7: Saving consensus hits with detailed statistics...")

# Save all hit sets
strict_hits.to_csv('screening_results/consensus_hits/strict_consensus_hits.csv', index=False)
majority_hits.to_csv('screening_results/consensus_hits/majority_consensus_hits.csv', index=False)
high_confidence_hits.to_csv('screening_results/consensus_hits/high_confidence_hits.csv', index=False)
very_high_confidence.to_csv('screening_results/consensus_hits/very_high_confidence_hits.csv', index=False)

# Create a summary of top hits
top_50_hits = results_df.nlargest(50, 'Consensus_Avg_Prob')[
    ['zincid', 'SMILES', 'Consensus_Avg_Prob', 'Consensus_Weighted_Prob', 
     'Consensus_Vote_Count', 'Morgan_RF_Prob', 'Morgan_XGB_Prob', 'Morgan_DNN_Prob',
     'MACCS_RF_Prob', 'MACCS_XGB_Prob', 'MACCS_DNN_Prob']
]
top_50_hits.to_csv('screening_results/consensus_hits/top_50_consensus_hits.csv', index=False)

# Add ranking
top_50_hits.insert(0, 'Rank', range(1, len(top_50_hits) + 1))
top_50_hits.to_csv('screening_results/consensus_hits/top_50_ranked_hits.csv', index=False)

print("✓ All hit sets saved to 'screening_results/consensus_hits/'")

# ============================================================================
# 8. GENERATE STATISTICAL REPORT
# ============================================================================
print("\nStep 8: Generating statistical report...")

# Calculate hit rates at different probability thresholds
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
hit_stats = []

for thresh in thresholds:
    hits = results_df[results_df['Consensus_Avg_Prob'] >= thresh]
    hit_stats.append({
        'Probability_Threshold': thresh,
        'Hits_Found': len(hits),
        'Hit_Rate_%': len(hits)/len(results_df)*100,
        'Avg_Vote_Count': hits['Consensus_Vote_Count'].mean() if len(hits) > 0 else 0
    })

hit_stats_df = pd.DataFrame(hit_stats)
hit_stats_df.to_csv('screening_results/hit_rate_statistics.csv', index=False)

# Model agreement analysis
agreement_matrix = pd.DataFrame({
    'Model_Pair': [
        'RF_Morgan vs XGB_Morgan', 'RF_Morgan vs DNN_Morgan', 'XGB_Morgan vs DNN_Morgan',
        'RF_MACCS vs XGB_MACCS', 'RF_MACCS vs DNN_MACCS', 'XGB_MACCS vs DNN_MACCS',
        'Morgan_avg vs MACCS_avg'
    ],
    'Agreement_%': [
        (results_df['Morgan_RF_Pred'] == results_df['Morgan_XGB_Pred']).mean() * 100,
        (results_df['Morgan_RF_Pred'] == results_df['Morgan_DNN_Pred']).mean() * 100,
        (results_df['Morgan_XGB_Pred'] == results_df['Morgan_DNN_Pred']).mean() * 100,
        (results_df['MACCS_RF_Pred'] == results_df['MACCS_XGB_Pred']).mean() * 100,
        (results_df['MACCS_RF_Pred'] == results_df['MACCS_DNN_Pred']).mean() * 100,
        (results_df['MACCS_XGB_Pred'] == results_df['MACCS_DNN_Pred']).mean() * 100,
        ((results_df[['Morgan_RF_Pred', 'Morgan_XGB_Pred', 'Morgan_DNN_Pred']].mean(axis=1) >= 0.5) ==
         (results_df[['MACCS_RF_Pred', 'MACCS_XGB_Pred', 'MACCS_DNN_Pred']].mean(axis=1) >= 0.5)).mean() * 100
    ]
})
agreement_matrix.to_csv('screening_results/model_agreement_analysis.csv', index=False)

# ============================================================================
# 9. CREATE SUMMARY REPORT
# ============================================================================
print("\nStep 9: Creating final summary report...")

summary_report = f"""
================================================================================
LDHA INHIBITOR VIRTUAL SCREENING SUMMARY REPORT
================================================================================
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

INPUT DATA:
- Total compounds screened: {len(results_df):,}
- Fingerprint types: Morgan (ECFP4, 2048 bits) and MACCS (167 bits)
- Models used: Random Forest, XGBoost, DNN (6 models total)

SCREENING RESULTS SUMMARY:
================================================================================
Hit Selection Strategy                    | Hits Found | Hit Rate (%)
--------------------------------------------------------------------------------
Strict Consensus (6/6 models)            | {len(strict_hits):>6,} | {len(strict_hits)/len(results_df)*100:>6.3f}%
Majority Consensus (≥4/6 models)         | {len(majority_hits):>6,} | {len(majority_hits)/len(results_df)*100:>6.3f}%
High Confidence (Prob>0.8 & Vote≥4)      | {len(high_confidence_hits):>6,} | {len(high_confidence_hits)/len(results_df)*100:>6.3f}%
Very High Confidence (Prob>0.9 & 6/6)    | {len(very_high_confidence):>6,} | {len(very_high_confidence)/len(results_df)*100:>6.3f}%
Top 10% by probability                   | {len(top_10_percent):>6,} | 10.000%
Top 50 ranked hits                       | 50 | 0.050%
================================================================================

MODEL PERFORMANCE ON SCREENING SET:
- Average probability across all compounds: {results_df['Consensus_Avg_Prob'].mean():.4f}
- Median probability: {results_df['Consensus_Avg_Prob'].median():.4f}
- Standard deviation: {results_df['Consensus_Avg_Prob'].std():.4f}

MODEL AGREEMENT:
- Best agreement pair: {agreement_matrix.loc[agreement_matrix['Agreement_%'].idxmax(), 'Model_Pair']}
  with {agreement_matrix['Agreement_%'].max():.1f}% agreement
- Average agreement across all model pairs: {agreement_matrix['Agreement_%'].mean():.1f}%

RECOMMENDED HITS FOR EXPERIMENTAL VALIDATION:
1. Primary candidates: {len(very_high_confidence)} compounds (Very High Confidence)
2. Secondary candidates: {len(high_confidence_hits) - len(very_high_confidence)} compounds (High Confidence)
3. Tertiary candidates: {len(majority_hits) - len(high_confidence_hits)} compounds (Majority Consensus)

OUTPUT FILES GENERATED:
================================================================================
1. screening_results/full_screening_results.csv.gz - Complete results (compressed)
2. screening_results/top_10_percent_hits.csv - Top 10% by consensus probability
3. screening_results/hit_rate_statistics.csv - Hit rates at various thresholds
4. screening_results/model_agreement_analysis.csv - Model agreement matrix
5. screening_results/consensus_hits/strict_consensus_hits.csv - 6/6 models active
6. screening_results/consensus_hits/majority_consensus_hits.csv - ≥4/6 models active
7. screening_results/consensus_hits/high_confidence_hits.csv - Prob>0.8 & Vote≥4
8. screening_results/consensus_hits/very_high_confidence_hits.csv - Prob>0.9 & 6/6
9. screening_results/consensus_hits/top_50_ranked_hits.csv - Top 50 ranked compounds
================================================================================

NEXT STEPS RECOMMENDED:
1. Perform molecular docking of top 50 consensus hits
2. Analyze chemical diversity of hit compounds
3. Select 10-20 diverse scaffolds for experimental testing
4. Perform ADMET prediction for top candidates
5. Validate top hits with molecular dynamics simulations

================================================================================
"""

# Save summary report
with open('screening_results/SCREENING_SUMMARY_REPORT.txt', 'w') as f:
    f.write(summary_report)

print(summary_report)

# ============================================================================
# 10. DISPLAY TOP 10 HITS
# ============================================================================
print("\n" + "="*80)
print("TOP 10 CONSENSUS HITS (By Average Probability)")
print("="*80)

top_10_display = top_50_hits.head(10)[['Rank', 'zincid', 'Consensus_Avg_Prob', 
                                        'Consensus_Vote_Count', 'Morgan_RF_Prob', 
                                        'MACCS_RF_Prob']].copy()
top_10_display['Consensus_Avg_Prob'] = top_10_display['Consensus_Avg_Prob'].round(4)
top_10_display['Morgan_RF_Prob'] = top_10_display['Morgan_RF_Prob'].round(4)
top_10_display['MACCS_RF_Prob'] = top_10_display['MACCS_RF_Prob'].round(4)

print(top_10_display.to_string(index=False))

print("\n" + "="*80)
print("✅ VIRTUAL SCREENING COMPLETED SUCCESSFULLY!")
print("="*80)
print("\n📁 All results saved in 'screening_results/' directory")
print("📁 Consensus hits saved in 'screening_results/consensus_hits/'")
print("\n🔬 Recommended hits for experimental validation:")
print(f"   - Very High Confidence: {len(very_high_confidence)} compounds")
print(f"   - High Confidence: {len(high_confidence_hits)} compounds")
print(f"   - Majority Consensus: {len(majority_hits)} compounds")

VIRTUAL SCREENING PIPELINE FOR LDHA INHIBITORS

Step 1: Loading trained models...
✓ All 6 models loaded successfully

Step 2: Loading screening data and fingerprints...
✓ Screening compounds: 100000
  Morgan fingerprints: (100000, 2048)
  MACCS fingerprints: (100000, 167)

Step 3: Making predictions with all models...
  Scaling Morgan fingerprints...
  Scaling MACCS fingerprints...

  Predicting with Morgan models...
  Predicting with MACCS models...
✓ Predictions completed for all 6 models

Step 4: Creating comprehensive results dataframe...

Step 5: Saving screening results...
✓ Full results saved: screening_results/full_screening_results.csv.gz
  File size: 7.33 MB
✓ Top 10% hits (10000 compounds) saved

Step 6: Identifying consensus hits...

  Strategy 1 - Strict Consensus (6/6 models):
    Hits found: 0 compounds
    Percentage: 0.000%

  Strategy 2 - Majority Consensus (≥4/6 models):
    Hits found: 527 compounds
    Percentage: 0.527%

  Strategy 3 - High Confidence (Prob>0.8 & 

UnicodeEncodeError: 'charmap' codec can't encode character '\u2265' in position 770: character maps to <undefined>

In [15]:
import pandas as pd
import numpy as np
import pickle
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, DataStructs
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
import os
from datetime import datetime
import tensorflow as tf

# Suppress TensorFlow warnings
tf.get_logger().setLevel('ERROR')

# Create directories for screening results
os.makedirs('screening_results', exist_ok=True)
os.makedirs('screening_results/consensus_hits', exist_ok=True)

# ============================================================================
# 1. LOAD ALL TRAINED MODELS
# ============================================================================
print("="*80)
print("VIRTUAL SCREENING PIPELINE FOR LDHA INHIBITORS")
print("="*80)
print("\nStep 1: Loading trained models...")

models = {}

# Load Morgan fingerprint models
models['Morgan_RF'] = pickle.load(open('models/Morgan_Random_Forest.pkl', 'rb'))
models['Morgan_XGB'] = pickle.load(open('models/Morgan_XGBoost.pkl', 'rb'))
models['Morgan_DNN'] = pickle.load(open('models/Morgan_DNN.pkl', 'rb'))
models['Morgan_Scaler'] = pickle.load(open('models/Morgan_Scaler.pkl', 'rb'))

# Load MACCS fingerprint models
models['MACCS_RF'] = pickle.load(open('models/MACCS_Random_Forest.pkl', 'rb'))
models['MACCS_XGB'] = pickle.load(open('models/MACCS_XGBoost.pkl', 'rb'))
models['MACCS_DNN'] = pickle.load(open('models/MACCS_DNN.pkl', 'rb'))
models['MACCS_Scaler'] = pickle.load(open('models/MACCS_Scaler.pkl', 'rb'))

print("✓ All 6 models loaded successfully")

# ============================================================================
# 2. LOAD SCREENING DATA AND FINGERPRINTS
# ============================================================================
print("\nStep 2: Loading screening data and fingerprints...")

# Load screening metadata
screening_metadata_morgan = pd.read_csv('data/screening_fingerprints/screening_morgan_metadata.csv')
screening_metadata_maccs = pd.read_csv('data/screening_fingerprints/screening_maccs_metadata.csv')

# Load fingerprints
screening_morgan_fps = np.load('data/screening_fingerprints/screening_morgan_fps.npy')
screening_maccs_fps = np.load('data/screening_fingerprints/screening_maccs_fps.npy')

print(f"✓ Screening compounds: {len(screening_metadata_morgan)}")
print(f"  Morgan fingerprints: {screening_morgan_fps.shape}")
print(f"  MACCS fingerprints: {screening_maccs_fps.shape}")

# ============================================================================
# 3. MAKE PREDICTIONS WITH ALL MODELS
# ============================================================================
print("\nStep 3: Making predictions with all models...")

# Scale Morgan fingerprints for DNN and XGBoost
print("  Scaling Morgan fingerprints...")
screening_morgan_scaled = models['Morgan_Scaler'].transform(screening_morgan_fps)

# Scale MACCS fingerprints
print("  Scaling MACCS fingerprints...")
screening_maccs_scaled = models['MACCS_Scaler'].transform(screening_maccs_fps)

# Initialize arrays for predictions
predictions = {}

# Morgan model predictions
print("\n  Predicting with Morgan models...")
predictions['Morgan_RF_proba'] = models['Morgan_RF'].predict_proba(screening_morgan_fps)[:, 1]
predictions['Morgan_XGB_proba'] = models['Morgan_XGB'].predict_proba(screening_morgan_scaled)[:, 1]
predictions['Morgan_DNN_proba'] = models['Morgan_DNN'].predict(screening_morgan_scaled, verbose=0).flatten()

# Morgan binary predictions (0.5 threshold)
predictions['Morgan_RF_pred'] = (predictions['Morgan_RF_proba'] >= 0.5).astype(int)
predictions['Morgan_XGB_pred'] = (predictions['Morgan_XGB_proba'] >= 0.5).astype(int)
predictions['Morgan_DNN_pred'] = (predictions['Morgan_DNN_proba'] >= 0.5).astype(int)

# MACCS model predictions
print("  Predicting with MACCS models...")
predictions['MACCS_RF_proba'] = models['MACCS_RF'].predict_proba(screening_maccs_fps)[:, 1]
predictions['MACCS_XGB_proba'] = models['MACCS_XGB'].predict_proba(screening_maccs_scaled)[:, 1]
predictions['MACCS_DNN_proba'] = models['MACCS_DNN'].predict(screening_maccs_scaled, verbose=0).flatten()

# MACCS binary predictions
predictions['MACCS_RF_pred'] = (predictions['MACCS_RF_proba'] >= 0.5).astype(int)
predictions['MACCS_XGB_pred'] = (predictions['MACCS_XGB_proba'] >= 0.5).astype(int)
predictions['MACCS_DNN_pred'] = (predictions['MACCS_DNN_proba'] >= 0.5).astype(int)

print("✓ Predictions completed for all 6 models")

# ============================================================================
# 4. CREATE COMPREHENSIVE RESULTS DATAFRAME
# ============================================================================
print("\nStep 4: Creating comprehensive results dataframe...")

# Use Morgan metadata as base (same compounds as MACCS)
results_df = screening_metadata_morgan.copy()

# Add all predictions
results_df['Morgan_RF_Prob'] = predictions['Morgan_RF_proba']
results_df['Morgan_XGB_Prob'] = predictions['Morgan_XGB_proba']
results_df['Morgan_DNN_Prob'] = predictions['Morgan_DNN_proba']
results_df['Morgan_RF_Pred'] = predictions['Morgan_RF_pred']
results_df['Morgan_XGB_Pred'] = predictions['Morgan_XGB_pred']
results_df['Morgan_DNN_Pred'] = predictions['Morgan_DNN_pred']

results_df['MACCS_RF_Prob'] = predictions['MACCS_RF_proba']
results_df['MACCS_XGB_Prob'] = predictions['MACCS_XGB_proba']
results_df['MACCS_DNN_Prob'] = predictions['MACCS_DNN_proba']
results_df['MACCS_RF_Pred'] = predictions['MACCS_RF_pred']
results_df['MACCS_XGB_Pred'] = predictions['MACCS_XGB_pred']
results_df['MACCS_DNN_Pred'] = predictions['MACCS_DNN_pred']

# Calculate consensus scores
# Consensus 1: Average probability across all 6 models
prob_columns = ['Morgan_RF_Prob', 'Morgan_XGB_Prob', 'Morgan_DNN_Prob',
                'MACCS_RF_Prob', 'MACCS_XGB_Prob', 'MACCS_DNN_Prob']
results_df['Consensus_Avg_Prob'] = results_df[prob_columns].mean(axis=1)

# Consensus 2: Weighted average (higher weight to better performing models)
weights = {
    'Morgan_RF_Prob': 0.20, 'Morgan_XGB_Prob': 0.19, 'Morgan_DNN_Prob': 0.18,
    'MACCS_RF_Prob': 0.15, 'MACCS_XGB_Prob': 0.16, 'MACCS_DNN_Prob': 0.12
}
results_df['Consensus_Weighted_Prob'] = sum(results_df[col] * weight for col, weight in weights.items())

# Consensus 3: Voting score (0-6, number of models predicting active)
vote_columns = ['Morgan_RF_Pred', 'Morgan_XGB_Pred', 'Morgan_DNN_Pred',
                'MACCS_RF_Pred', 'MACCS_XGB_Pred', 'MACCS_DNN_Pred']
results_df['Consensus_Vote_Count'] = results_df[vote_columns].sum(axis=1)

# Consensus 4: Strict consensus (all models agree)
results_df['Strict_Consensus'] = (results_df['Consensus_Vote_Count'] == 6).astype(int)

# Consensus 5: Majority consensus (>=4 models agree)
results_df['Majority_Consensus'] = (results_df['Consensus_Vote_Count'] >= 4).astype(int)

# ============================================================================
# 5. SAVE COMPLETE SCREENING RESULTS
# ============================================================================
print("\nStep 5: Saving screening results...")

# Save full results (compressed for large file)
results_df.to_csv('screening_results/full_screening_results.csv.gz', 
                  index=False, compression='gzip')
print(f"✓ Full results saved: screening_results/full_screening_results.csv.gz")
print(f"  File size: {os.path.getsize('screening_results/full_screening_results.csv.gz') / 1e6:.2f} MB")

# Save top 10% by consensus probability
top_10_percent = results_df.nlargest(int(len(results_df) * 0.1), 'Consensus_Avg_Prob')
top_10_percent.to_csv('screening_results/top_10_percent_hits.csv', index=False)
print(f"✓ Top 10% hits ({len(top_10_percent)} compounds) saved")

# ============================================================================
# 6. IDENTIFY CONSENSUS HITS (Active by all models)
# ============================================================================
print("\nStep 6: Identifying consensus hits...")

# Strategy 1: Strict consensus (all 6 models predict active)
strict_hits = results_df[results_df['Strict_Consensus'] == 1].copy()
print(f"\n  Strategy 1 - Strict Consensus (6/6 models):")
print(f"    Hits found: {len(strict_hits)} compounds")
print(f"    Percentage: {len(strict_hits)/len(results_df)*100:.3f}%")

# Strategy 2: Majority consensus (>=4 models predict active)
majority_hits = results_df[results_df['Majority_Consensus'] == 1].copy()
print(f"\n  Strategy 2 - Majority Consensus (>=4/6 models):")
print(f"    Hits found: {len(majority_hits)} compounds")
print(f"    Percentage: {len(majority_hits)/len(results_df)*100:.3f}%")

# Strategy 3: High confidence hits (Avg probability > 0.8 AND vote count >= 4)
high_confidence_hits = results_df[
    (results_df['Consensus_Avg_Prob'] > 0.8) & 
    (results_df['Consensus_Vote_Count'] >= 4)
].copy()
print(f"\n  Strategy 3 - High Confidence (Prob>0.8 & Vote>=4):")
print(f"    Hits found: {len(high_confidence_hits)} compounds")
print(f"    Percentage: {len(high_confidence_hits)/len(results_df)*100:.3f}%")

# Strategy 4: Very high confidence (Prob > 0.9 AND all models agree)
very_high_confidence = results_df[
    (results_df['Consensus_Avg_Prob'] > 0.9) & 
    (results_df['Strict_Consensus'] == 1)
].copy()
print(f"\n  Strategy 4 - Very High Confidence (Prob>0.9 & 6/6):")
print(f"    Hits found: {len(very_high_confidence)} compounds")
print(f"    Percentage: {len(very_high_confidence)/len(results_df)*100:.3f}%")

# ============================================================================
# 7. SAVE CONSENSUS HITS WITH STATISTICS
# ============================================================================
print("\nStep 7: Saving consensus hits with detailed statistics...")

# Save all hit sets
if len(strict_hits) > 0:
    strict_hits.to_csv('screening_results/consensus_hits/strict_consensus_hits.csv', index=False)
if len(majority_hits) > 0:
    majority_hits.to_csv('screening_results/consensus_hits/majority_consensus_hits.csv', index=False)
if len(high_confidence_hits) > 0:
    high_confidence_hits.to_csv('screening_results/consensus_hits/high_confidence_hits.csv', index=False)
if len(very_high_confidence) > 0:
    very_high_confidence.to_csv('screening_results/consensus_hits/very_high_confidence_hits.csv', index=False)

# Create a summary of top hits
top_50_hits = results_df.nlargest(50, 'Consensus_Avg_Prob')[
    ['zincid', 'SMILES', 'Consensus_Avg_Prob', 'Consensus_Weighted_Prob', 
     'Consensus_Vote_Count', 'Morgan_RF_Prob', 'Morgan_XGB_Prob', 'Morgan_DNN_Prob',
     'MACCS_RF_Prob', 'MACCS_XGB_Prob', 'MACCS_DNN_Prob']
]
top_50_hits.to_csv('screening_results/consensus_hits/top_50_consensus_hits.csv', index=False)

# Add ranking
top_50_hits.insert(0, 'Rank', range(1, len(top_50_hits) + 1))
top_50_hits.to_csv('screening_results/consensus_hits/top_50_ranked_hits.csv', index=False)

print("✓ All hit sets saved to 'screening_results/consensus_hits/'")

# ============================================================================
# 8. GENERATE STATISTICAL REPORT
# ============================================================================
print("\nStep 8: Generating statistical report...")

# Calculate hit rates at different probability thresholds
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
hit_stats = []

for thresh in thresholds:
    hits = results_df[results_df['Consensus_Avg_Prob'] >= thresh]
    hit_stats.append({
        'Probability_Threshold': thresh,
        'Hits_Found': len(hits),
        'Hit_Rate_Percent': len(hits)/len(results_df)*100,
        'Avg_Vote_Count': hits['Consensus_Vote_Count'].mean() if len(hits) > 0 else 0
    })

hit_stats_df = pd.DataFrame(hit_stats)
hit_stats_df.to_csv('screening_results/hit_rate_statistics.csv', index=False)

# Model agreement analysis
agreement_matrix = pd.DataFrame({
    'Model_Pair': [
        'RF_Morgan vs XGB_Morgan', 'RF_Morgan vs DNN_Morgan', 'XGB_Morgan vs DNN_Morgan',
        'RF_MACCS vs XGB_MACCS', 'RF_MACCS vs DNN_MACCS', 'XGB_MACCS vs DNN_MACCS',
        'Morgan_avg vs MACCS_avg'
    ],
    'Agreement_Percent': [
        (results_df['Morgan_RF_Pred'] == results_df['Morgan_XGB_Pred']).mean() * 100,
        (results_df['Morgan_RF_Pred'] == results_df['Morgan_DNN_Pred']).mean() * 100,
        (results_df['Morgan_XGB_Pred'] == results_df['Morgan_DNN_Pred']).mean() * 100,
        (results_df['MACCS_RF_Pred'] == results_df['MACCS_XGB_Pred']).mean() * 100,
        (results_df['MACCS_RF_Pred'] == results_df['MACCS_DNN_Pred']).mean() * 100,
        (results_df['MACCS_XGB_Pred'] == results_df['MACCS_DNN_Pred']).mean() * 100,
        ((results_df[['Morgan_RF_Pred', 'Morgan_XGB_Pred', 'Morgan_DNN_Pred']].mean(axis=1) >= 0.5) ==
         (results_df[['MACCS_RF_Pred', 'MACCS_XGB_Pred', 'MACCS_DNN_Pred']].mean(axis=1) >= 0.5)).mean() * 100
    ]
})
agreement_matrix.to_csv('screening_results/model_agreement_analysis.csv', index=False)

# ============================================================================
# 9. CREATE SUMMARY REPORT (WITHOUT UNICODE ISSUES)
# ============================================================================
print("\nStep 9: Creating final summary report...")

summary_report = f"""
================================================================================
LDHA INHIBITOR VIRTUAL SCREENING SUMMARY REPORT
================================================================================
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

INPUT DATA:
- Total compounds screened: {len(results_df):,}
- Fingerprint types: Morgan (ECFP4, 2048 bits) and MACCS (167 bits)
- Models used: Random Forest, XGBoost, DNN (6 models total)

SCREENING RESULTS SUMMARY:
================================================================================
Hit Selection Strategy                    | Hits Found | Hit Rate (%)
--------------------------------------------------------------------------------
Strict Consensus (6/6 models)            | {len(strict_hits):>6,} | {len(strict_hits)/len(results_df)*100:>6.3f}%
Majority Consensus (>=4/6 models)        | {len(majority_hits):>6,} | {len(majority_hits)/len(results_df)*100:>6.3f}%
High Confidence (Prob>0.8 & Vote>=4)     | {len(high_confidence_hits):>6,} | {len(high_confidence_hits)/len(results_df)*100:>6.3f}%
Very High Confidence (Prob>0.9 & 6/6)    | {len(very_high_confidence):>6,} | {len(very_high_confidence)/len(results_df)*100:>6.3f}%
Top 10% by probability                   | {len(top_10_percent):>6,} | 10.000%
Top 50 ranked hits                       | 50 | 0.050%
================================================================================

MODEL PERFORMANCE ON SCREENING SET:
- Average probability across all compounds: {results_df['Consensus_Avg_Prob'].mean():.4f}
- Median probability: {results_df['Consensus_Avg_Prob'].median():.4f}
- Standard deviation: {results_df['Consensus_Avg_Prob'].std():.4f}
- 95th percentile probability: {results_df['Consensus_Avg_Prob'].quantile(0.95):.4f}

MODEL AGREEMENT:
- Best agreement pair: {agreement_matrix.loc[agreement_matrix['Agreement_Percent'].idxmax(), 'Model_Pair']}
  with {agreement_matrix['Agreement_Percent'].max():.1f}% agreement
- Average agreement across all model pairs: {agreement_matrix['Agreement_Percent'].mean():.1f}%

RECOMMENDED HITS FOR EXPERIMENTAL VALIDATION:
1. Primary candidates: {len(very_high_confidence)} compounds (Very High Confidence)
2. Secondary candidates: {len(high_confidence_hits) - len(very_high_confidence)} compounds (High Confidence)
3. Tertiary candidates: {len(majority_hits) - len(high_confidence_hits)} compounds (Majority Consensus)

OUTPUT FILES GENERATED:
================================================================================
1. screening_results/full_screening_results.csv.gz - Complete results (compressed)
2. screening_results/top_10_percent_hits.csv - Top 10% by consensus probability
3. screening_results/hit_rate_statistics.csv - Hit rates at various thresholds
4. screening_results/model_agreement_analysis.csv - Model agreement matrix
5. screening_results/consensus_hits/majority_consensus_hits.csv - >=4/6 models active
6. screening_results/consensus_hits/top_50_ranked_hits.csv - Top 50 ranked compounds

NOTE: Strict consensus and high confidence hits were not found at current thresholds.

NEXT STEPS RECOMMENDED:
1. Perform molecular docking of top 50 consensus hits
2. Analyze chemical diversity of hit compounds
3. Select 10-20 diverse scaffolds for experimental testing
4. Perform ADMET prediction for top candidates
5. Validate top hits with molecular dynamics simulations

================================================================================
"""

# Save summary report with proper encoding
with open('screening_results/SCREENING_SUMMARY_REPORT.txt', 'w', encoding='utf-8') as f:
    f.write(summary_report)

print("✓ Summary report saved")

# ============================================================================
# 10. DISPLAY TOP 10 HITS
# ============================================================================
print("\n" + "="*80)
print("TOP 10 CONSENSUS HITS (By Average Probability)")
print("="*80)

top_10_display = top_50_hits.head(10)[['Rank', 'zincid', 'Consensus_Avg_Prob', 
                                        'Consensus_Vote_Count', 'Morgan_RF_Prob', 
                                        'MACCS_RF_Prob']].copy()
top_10_display['Consensus_Avg_Prob'] = top_10_display['Consensus_Avg_Prob'].round(4)
top_10_display['Morgan_RF_Prob'] = top_10_display['Morgan_RF_Prob'].round(4)
top_10_display['MACCS_RF_Prob'] = top_10_display['MACCS_RF_Prob'].round(4)

print(top_10_display.to_string(index=False))

print("\n" + "="*80)
print("VIRTUAL SCREENING COMPLETED SUCCESSFULLY!")
print("="*80)
print("\n📁 All results saved in 'screening_results/' directory")
print("📁 Consensus hits saved in 'screening_results/consensus_hits/'")
print("\n🔬 Recommended hits for next steps:")
print(f"   - Majority Consensus Hits: {len(majority_hits)} compounds")
print(f"   - Top 50 Ranked Hits: 50 compounds for docking studies")

VIRTUAL SCREENING PIPELINE FOR LDHA INHIBITORS

Step 1: Loading trained models...
✓ All 6 models loaded successfully

Step 2: Loading screening data and fingerprints...
✓ Screening compounds: 100000
  Morgan fingerprints: (100000, 2048)
  MACCS fingerprints: (100000, 167)

Step 3: Making predictions with all models...
  Scaling Morgan fingerprints...
  Scaling MACCS fingerprints...

  Predicting with Morgan models...
  Predicting with MACCS models...
✓ Predictions completed for all 6 models

Step 4: Creating comprehensive results dataframe...

Step 5: Saving screening results...
✓ Full results saved: screening_results/full_screening_results.csv.gz
  File size: 7.33 MB
✓ Top 10% hits (10000 compounds) saved

Step 6: Identifying consensus hits...

  Strategy 1 - Strict Consensus (6/6 models):
    Hits found: 0 compounds
    Percentage: 0.000%

  Strategy 2 - Majority Consensus (>=4/6 models):
    Hits found: 527 compounds
    Percentage: 0.527%

  Strategy 3 - High Confidence (Prob>0.8 &

In [16]:
import pandas as pd
import numpy as np
import pickle
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, DataStructs
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
import os
from datetime import datetime
import tensorflow as tf

# Suppress TensorFlow warnings
tf.get_logger().setLevel('ERROR')

# Create directories for results
os.makedirs('Hit-compounds', exist_ok=True)
os.makedirs('screening_results', exist_ok=True)

# ============================================================================
# STEP 1: LOAD ALL TRAINED MODELS
# ============================================================================
print("="*80)
print("VIRTUAL SCREENING PIPELINE FOR LDHA INHIBITORS")
print("="*80)
print("\nStep 1: Loading trained models...")

models = {}

# Load Morgan fingerprint models
models['Morgan_RF'] = pickle.load(open('models/Morgan_Random_Forest.pkl', 'rb'))
models['Morgan_XGB'] = pickle.load(open('models/Morgan_XGBoost.pkl', 'rb'))
models['Morgan_DNN'] = pickle.load(open('models/Morgan_DNN.pkl', 'rb'))
models['Morgan_Scaler'] = pickle.load(open('models/Morgan_Scaler.pkl', 'rb'))

# Load MACCS fingerprint models
models['MACCS_RF'] = pickle.load(open('models/MACCS_Random_Forest.pkl', 'rb'))
models['MACCS_XGB'] = pickle.load(open('models/MACCS_XGBoost.pkl', 'rb'))
models['MACCS_DNN'] = pickle.load(open('models/MACCS_DNN.pkl', 'rb'))
models['MACCS_Scaler'] = pickle.load(open('models/MACCS_Scaler.pkl', 'rb'))

print("✓ All 6 models loaded successfully")

# ============================================================================
# STEP 2: LOAD SCREENING DATA AND FINGERPRINTS
# ============================================================================
print("\nStep 2: Loading screening data and fingerprints...")

# Load screening metadata
screening_metadata_morgan = pd.read_csv('data/screening_fingerprints/screening_morgan_metadata.csv')
screening_metadata_maccs = pd.read_csv('data/screening_fingerprints/screening_maccs_metadata.csv')

# Load fingerprints
screening_morgan_fps = np.load('data/screening_fingerprints/screening_morgan_fps.npy')
screening_maccs_fps = np.load('data/screening_fingerprints/screening_maccs_fps.npy')

print(f"✓ Screening compounds: {len(screening_metadata_morgan):,}")
print(f"  Morgan fingerprints: {screening_morgan_fps.shape}")
print(f"  MACCS fingerprints: {screening_maccs_fps.shape}")

# ============================================================================
# STEP 3: MAKE PREDICTIONS WITH ALL MODELS
# ============================================================================
print("\nStep 3: Making predictions with all models...")

# Scale fingerprints
print("  Scaling fingerprints...")
screening_morgan_scaled = models['Morgan_Scaler'].transform(screening_morgan_fps)
screening_maccs_scaled = models['MACCS_Scaler'].transform(screening_maccs_fps)

# Initialize arrays for predictions
predictions = {}

# Morgan model predictions
print("\n  Predicting with Morgan models...")
predictions['Morgan_RF_proba'] = models['Morgan_RF'].predict_proba(screening_morgan_fps)[:, 1]
predictions['Morgan_XGB_proba'] = models['Morgan_XGB'].predict_proba(screening_morgan_scaled)[:, 1]
predictions['Morgan_DNN_proba'] = models['Morgan_DNN'].predict(screening_morgan_scaled, verbose=0).flatten()

# Morgan binary predictions (0.5 threshold)
predictions['Morgan_RF_pred'] = (predictions['Morgan_RF_proba'] >= 0.5).astype(int)
predictions['Morgan_XGB_pred'] = (predictions['Morgan_XGB_proba'] >= 0.5).astype(int)
predictions['Morgan_DNN_pred'] = (predictions['Morgan_DNN_proba'] >= 0.5).astype(int)

# MACCS model predictions
print("  Predicting with MACCS models...")
predictions['MACCS_RF_proba'] = models['MACCS_RF'].predict_proba(screening_maccs_fps)[:, 1]
predictions['MACCS_XGB_proba'] = models['MACCS_XGB'].predict_proba(screening_maccs_scaled)[:, 1]
predictions['MACCS_DNN_proba'] = models['MACCS_DNN'].predict(screening_maccs_scaled, verbose=0).flatten()

# MACCS binary predictions
predictions['MACCS_RF_pred'] = (predictions['MACCS_RF_proba'] >= 0.5).astype(int)
predictions['MACCS_XGB_pred'] = (predictions['MACCS_XGB_proba'] >= 0.5).astype(int)
predictions['MACCS_DNN_pred'] = (predictions['MACCS_DNN_proba'] >= 0.5).astype(int)

print("✓ Predictions completed for all 6 models")

# ============================================================================
# STEP 4: IDENTIFY AND SAVE MAJORITY CONSENSUS HITS (>=4 MODELS ACTIVE)
# ============================================================================
print("\nStep 4: Identifying majority consensus hits (>=4 models active)...")

# Create results dataframe
results_df = screening_metadata_morgan.copy()

# Add all predictions
results_df['Morgan_RF_Prob'] = predictions['Morgan_RF_proba']
results_df['Morgan_XGB_Prob'] = predictions['Morgan_XGB_proba']
results_df['Morgan_DNN_Prob'] = predictions['Morgan_DNN_proba']
results_df['Morgan_RF_Pred'] = predictions['Morgan_RF_pred']
results_df['Morgan_XGB_Pred'] = predictions['Morgan_XGB_pred']
results_df['Morgan_DNN_Pred'] = predictions['Morgan_DNN_pred']

results_df['MACCS_RF_Prob'] = predictions['MACCS_RF_proba']
results_df['MACCS_XGB_Prob'] = predictions['MACCS_XGB_proba']
results_df['MACCS_DNN_Prob'] = predictions['MACCS_DNN_proba']
results_df['MACCS_RF_Pred'] = predictions['MACCS_RF_pred']
results_df['MACCS_XGB_Pred'] = predictions['MACCS_XGB_pred']
results_df['MACCS_DNN_Pred'] = predictions['MACCS_DNN_pred']

# Calculate vote count (number of models predicting active)
vote_columns = ['Morgan_RF_Pred', 'Morgan_XGB_Pred', 'Morgan_DNN_Pred',
                'MACCS_RF_Pred', 'MACCS_XGB_Pred', 'MACCS_DNN_Pred']
results_df['Vote_Count'] = results_df[vote_columns].sum(axis=1)

# Calculate consensus probability (average of all 6 models)
prob_columns = ['Morgan_RF_Prob', 'Morgan_XGB_Prob', 'Morgan_DNN_Prob',
                'MACCS_RF_Prob', 'MACCS_XGB_Prob', 'MACCS_DNN_Prob']
results_df['Consensus_Probability'] = results_df[prob_columns].mean(axis=1)

# Identify majority consensus hits (>=4 models predict active)
majority_hits = results_df[results_df['Vote_Count'] >= 4].copy()

print(f"\n  Total compounds screened: {len(results_df):,}")
print(f"  Majority consensus hits (>=4 models): {len(majority_hits):,}")
print(f"  Hit rate: {len(majority_hits)/len(results_df)*100:.3f}%")

# Sort by consensus probability (highest first)
majority_hits = majority_hits.sort_values('Consensus_Probability', ascending=False)

# Add rank column
majority_hits.insert(0, 'Rank', range(1, len(majority_hits) + 1))

# Select relevant columns for output
output_columns = ['Rank', 'zincid', 'SMILES', 'Consensus_Probability', 'Vote_Count',
                  'Morgan_RF_Prob', 'Morgan_XGB_Prob', 'Morgan_DNN_Prob',
                  'MACCS_RF_Prob', 'MACCS_XGB_Prob', 'MACCS_DNN_Prob',
                  'Morgan_RF_Pred', 'Morgan_XGB_Pred', 'Morgan_DNN_Pred',
                  'MACCS_RF_Pred', 'MACCS_XGB_Pred', 'MACCS_DNN_Pred']

majority_hits_output = majority_hits[output_columns]

# Round probability columns for better readability
prob_cols = ['Consensus_Probability', 'Morgan_RF_Prob', 'Morgan_XGB_Prob', 
             'Morgan_DNN_Prob', 'MACCS_RF_Prob', 'MACCS_XGB_Prob', 'MACCS_DNN_Prob']
majority_hits_output[prob_cols] = majority_hits_output[prob_cols].round(6)

# ============================================================================
# SAVE RESULTS TO Hit-compounds FOLDER
# ============================================================================
print("\nStep 5: Saving results to 'Hit-compounds' folder...")

# Save all majority consensus hits
majority_hits_output.to_csv('Hit-compounds/majority_consensus_hits.csv', index=False)
print(f"✓ Saved: Hit-compounds/majority_consensus_hits.csv ({len(majority_hits)} compounds)")

# Save top 10, 25, 50, 100 hits for easy access
top_n = [10, 25, 50, 100]
for n in top_n:
    if n <= len(majority_hits):
        top_hits = majority_hits_output.head(n)
        top_hits.to_csv(f'Hit-compounds/top_{n}_hits.csv', index=False)
        print(f"✓ Saved: Hit-compounds/top_{n}_hits.csv")

# Save hits with vote count = 6 (if any)
strict_hits = majority_hits[majority_hits['Vote_Count'] == 6]
if len(strict_hits) > 0:
    strict_hits_output = strict_hits[output_columns]
    strict_hits_output.to_csv('Hit-compounds/strict_consensus_hits_6_6.csv', index=False)
    print(f"✓ Saved: Hit-compounds/strict_consensus_hits_6_6.csv ({len(strict_hits)} compounds)")
else:
    print("  Note: No compounds with strict consensus (6/6 models) found")

# Save hits with vote count = 5
vote5_hits = majority_hits[majority_hits['Vote_Count'] == 5]
if len(vote5_hits) > 0:
    vote5_output = vote5_hits[output_columns]
    vote5_output.to_csv('Hit-compounds/vote_count_5_hits.csv', index=False)
    print(f"✓ Saved: Hit-compounds/vote_count_5_hits.csv ({len(vote5_hits)} compounds)")

# Save hits with vote count = 4
vote4_hits = majority_hits[majority_hits['Vote_Count'] == 4]
if len(vote4_hits) > 0:
    vote4_output = vote4_hits[output_columns]
    vote4_output.to_csv('Hit-compounds/vote_count_4_hits.csv', index=False)
    print(f"✓ Saved: Hit-compounds/vote_count_4_hits.csv ({len(vote4_hits)} compounds)")

# ============================================================================
# CREATE SUMMARY REPORT
# ============================================================================
print("\nStep 6: Creating summary report...")

summary_report = f"""
================================================================================
LDHA INHIBITOR SCREENING - MAJORITY CONSENSUS HITS SUMMARY
================================================================================
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

SCREENING STATISTICS:
--------------------------------------------------------------------------------
Total compounds screened:                 {len(results_df):,}
Majority consensus hits (>=4 models):     {len(majority_hits):,} ({len(majority_hits)/len(results_df)*100:.3f}%)
  - Compounds with 6/6 models active:     {len(strict_hits)} ({len(strict_hits)/len(results_df)*100:.3f}%)
  - Compounds with 5/6 models active:     {len(vote5_hits)} ({len(vote5_hits)/len(results_df)*100:.3f}%)
  - Compounds with 4/6 models active:     {len(vote4_hits)} ({len(vote4_hits)/len(results_df)*100:.3f}%)

CONSENSUS PROBABILITY DISTRIBUTION:
--------------------------------------------------------------------------------
Minimum consensus probability:            {majority_hits['Consensus_Probability'].min():.6f}
Maximum consensus probability:            {majority_hits['Consensus_Probability'].max():.6f}
Mean consensus probability:               {majority_hits['Consensus_Probability'].mean():.6f}
Median consensus probability:             {majority_hits['Consensus_Probability'].median():.6f}

TOP 10 HITS (By Consensus Probability):
--------------------------------------------------------------------------------
"""

# Add top 10 hits to summary
top_10 = majority_hits_output.head(10)
for idx, row in top_10.iterrows():
    summary_report += f"Rank {int(row['Rank'])}: {row['zincid']} | Consensus Prob: {row['Consensus_Probability']:.6f} | Vote Count: {int(row['Vote_Count'])}\n"

summary_report += f"""
================================================================================
FILES SAVED IN 'Hit-compounds' FOLDER:
================================================================================
1. majority_consensus_hits.csv          - All {len(majority_hits)} hits with vote count >=4
2. top_10_hits.csv                      - Top 10 ranked hits
3. top_25_hits.csv                      - Top 25 ranked hits
4. top_50_hits.csv                      - Top 50 ranked hits
5. top_100_hits.csv                     - Top 100 ranked hits
6. vote_count_5_hits.csv                - {len(vote5_hits)} compounds with 5/6 models active
7. vote_count_4_hits.csv                - {len(vote4_hits)} compounds with 4/6 models active
"""

if len(strict_hits) > 0:
    summary_report += f"8. strict_consensus_hits_6_6.csv        - {len(strict_hits)} compounds with 6/6 models active\n"

summary_report += f"""
================================================================================
RECOMMENDED NEXT STEPS:
================================================================================
1. Review top 50 hits for chemical diversity and drug-likeness
2. Perform molecular docking of top 50 hits with LDHA active site
3. Select 10-20 diverse compounds for experimental validation
4. Perform ADMET prediction for selected candidates
5. In vitro LDHA inhibition assays for top candidates

================================================================================
"""

# Save summary report
with open('Hit-compounds/SCREENING_SUMMARY.txt', 'w', encoding='utf-8') as f:
    f.write(summary_report)

print("✓ Saved: Hit-compounds/SCREENING_SUMMARY.txt")

# ============================================================================
# DISPLAY RESULTS
# ============================================================================
print("\n" + "="*80)
print("SCREENING COMPLETE - MAJORITY CONSENSUS HITS")
print("="*80)
print(f"\nTotal hits identified: {len(majority_hits)} compounds")
print(f"Hit rate: {len(majority_hits)/len(results_df)*100:.3f}%")
print(f"\nVote count distribution:")
print(f"  - 6/6 models: {len(strict_hits)} compounds")
print(f"  - 5/6 models: {len(vote5_hits)} compounds")
print(f"  - 4/6 models: {len(vote4_hits)} compounds")

print("\n" + "-"*80)
print("TOP 10 RANKED HITS (Highest Consensus Probability)")
print("-"*80)
print(top_10[['Rank', 'zincid', 'Consensus_Probability', 'Vote_Count']].to_string(index=False))

print("\n" + "="*80)
print("RESULTS SAVED IN 'Hit-compounds' FOLDER")
print("="*80)
print("""
Files available:
  - majority_consensus_hits.csv (All hits with complete data)
  - top_10_hits.csv to top_100_hits.csv (Prioritized hits)
  - vote_count_4_hits.csv and vote_count_5_hits.csv (Filtered by model agreement)
  - SCREENING_SUMMARY.txt (Complete report)

Ready for next step: Molecular docking and experimental validation!
""")

VIRTUAL SCREENING PIPELINE FOR LDHA INHIBITORS

Step 1: Loading trained models...
✓ All 6 models loaded successfully

Step 2: Loading screening data and fingerprints...
✓ Screening compounds: 100,000
  Morgan fingerprints: (100000, 2048)
  MACCS fingerprints: (100000, 167)

Step 3: Making predictions with all models...
  Scaling fingerprints...

  Predicting with Morgan models...
  Predicting with MACCS models...
✓ Predictions completed for all 6 models

Step 4: Identifying majority consensus hits (>=4 models active)...

  Total compounds screened: 100,000
  Majority consensus hits (>=4 models): 527
  Hit rate: 0.527%

Step 5: Saving results to 'Hit-compounds' folder...
✓ Saved: Hit-compounds/majority_consensus_hits.csv (527 compounds)
✓ Saved: Hit-compounds/top_10_hits.csv
✓ Saved: Hit-compounds/top_25_hits.csv
✓ Saved: Hit-compounds/top_50_hits.csv
✓ Saved: Hit-compounds/top_100_hits.csv
  Note: No compounds with strict consensus (6/6 models) found
✓ Saved: Hit-compounds/vote_count_5

In [18]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

# Suppress TensorFlow warnings
tf.get_logger().setLevel('ERROR')

import os
os.makedirs('5-Hits', exist_ok=True)
os.makedirs('5-Hits/predictions', exist_ok=True)
os.makedirs('5-Hits/hits', exist_ok=True)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_models_and_scalers():
    """Load all trained models and scalers"""
    models = {}
    
    print("Loading trained models...")
    
    # Morgan models
    models['RF_Morgan'] = pickle.load(open('models/Morgan_Random_Forest.pkl', 'rb'))
    models['XGB_Morgan'] = pickle.load(open('models/Morgan_XGBoost.pkl', 'rb'))
    models['DNN_Morgan'] = pickle.load(open('models/Morgan_DNN.pkl', 'rb'))
    
    # MACCS models
    models['RF_MACCS'] = pickle.load(open('models/MACCS_Random_Forest.pkl', 'rb'))
    models['XGB_MACCS'] = pickle.load(open('models/MACCS_XGBoost.pkl', 'rb'))
    models['DNN_MACCS'] = pickle.load(open('models/MACCS_DNN.pkl', 'rb'))
    
    # Load scalers
    scaler_morgan = pickle.load(open('models/Morgan_Scaler.pkl', 'rb'))
    scaler_maccs = pickle.load(open('models/MACCS_Scaler.pkl', 'rb'))
    
    print("  ✓ All 6 models and scalers loaded successfully")
    
    return models, scaler_morgan, scaler_maccs

def predict_with_consensus(models, scaler_morgan, scaler_maccs, 
                          morgan_fps, maccs_fps, metadata):
    """
    Make predictions using all models and calculate consensus scores
    """
    print("\nPerforming predictions with all 6 models...")
    
    # Scale fingerprints
    print("  Scaling fingerprints...")
    morgan_scaled = scaler_morgan.transform(morgan_fps)
    maccs_scaled = scaler_maccs.transform(maccs_fps)
    
    # Initialize arrays for predictions
    n_compounds = len(morgan_fps)
    predictions = {}
    
    # Random Forest predictions
    print("  Random Forest (Morgan)...")
    predictions['RF_Morgan'] = models['RF_Morgan'].predict_proba(morgan_fps)[:, 1]
    
    print("  Random Forest (MACCS)...")
    predictions['RF_MACCS'] = models['RF_MACCS'].predict_proba(maccs_fps)[:, 1]
    
    # XGBoost predictions
    print("  XGBoost (Morgan)...")
    predictions['XGB_Morgan'] = models['XGB_Morgan'].predict_proba(morgan_scaled)[:, 1]
    
    print("  XGBoost (MACCS)...")
    predictions['XGB_MACCS'] = models['XGB_MACCS'].predict_proba(maccs_scaled)[:, 1]
    
    # DNN predictions
    print("  DNN (Morgan)...")
    predictions['DNN_Morgan'] = models['DNN_Morgan'].predict(morgan_scaled, verbose=0).flatten()
    
    print("  DNN (MACCS)...")
    predictions['DNN_MACCS'] = models['DNN_MACCS'].predict(maccs_scaled, verbose=0).flatten()
    
    # Calculate consensus scores
    print("\nCalculating consensus scores...")
    
    # Method 1: Average of all 6 models
    predictions['Consensus_Avg'] = np.mean([predictions[k] for k in predictions.keys()], axis=0)
    
    # Method 2: Weighted average (higher weight to better performing models)
    weights = {
        'RF_Morgan': 1.0,      # AUC: 0.9887
        'XGB_Morgan': 0.95,    # AUC: 0.9842
        'DNN_Morgan': 0.95,    # AUC: 0.9831
        'RF_MACCS': 0.92,      # AUC: 0.9770
        'XGB_MACCS': 0.92,     # AUC: 0.9787
        'DNN_MACCS': 0.90      # AUC: 0.9674
    }
    
    weighted_sum = np.zeros(n_compounds)
    total_weight = sum(weights.values())
    
    for model_name, weight in weights.items():
        weighted_sum += predictions[model_name] * weight
    
    predictions['Consensus_Weighted'] = weighted_sum / total_weight
    
    # Method 3: Voting (at least 4 models predict active > 0.5)
    votes = np.zeros((n_compounds, len(predictions)))
    for idx, (model_name, pred) in enumerate(predictions.items()):
        if not model_name.startswith('Consensus'):
            votes[:, idx] = (pred > 0.5).astype(int)
    
    predictions['Consensus_Votes'] = np.sum(votes, axis=1)
    
    # Method 4: Geometric mean of probabilities
    prob_product = np.ones(n_compounds)
    for model_name, pred in predictions.items():
        if not model_name.startswith('Consensus'):
            prob_product *= pred
    
    predictions['Consensus_Geometric'] = prob_product ** (1/6)
    
    return predictions

def identify_hits(predictions, metadata, thresholds):
    """
    Identify high-confidence hits based on multiple criteria
    """
    print("\n" + "="*70)
    print("IDENTIFYING HIGH-CONFIDENCE HITS")
    print("="*70)
    
    # Create results dataframe
    results_df = metadata.copy()
    
    # Add all predictions
    for model_name, pred in predictions.items():
        results_df[model_name] = pred
    
    # Add binary predictions (threshold = 0.5)
    for model_name in ['RF_Morgan', 'XGB_Morgan', 'DNN_Morgan', 
                      'RF_MACCS', 'XGB_MACCS', 'DNN_MACCS']:
        results_df[f'{model_name}_Binary'] = (results_df[model_name] > 0.5).astype(int)
    
    # Calculate additional metrics
    results_df['Models_Active_Count'] = results_df[[f'{m}_Binary' for m in ['RF_Morgan', 'XGB_Morgan', 'DNN_Morgan', 
                                                                              'RF_MACCS', 'XGB_MACCS', 'DNN_MACCS']]].sum(axis=1)
    
    results_df['Consensus_Active'] = (results_df['Consensus_Avg'] > thresholds['consensus_avg']).astype(int)
    results_df['Consensus_Weighted_Active'] = (results_df['Consensus_Weighted'] > thresholds['consensus_weighted']).astype(int)
    results_df['Vote_Active'] = (results_df['Consensus_Votes'] >= thresholds['min_votes']).astype(int)
    
    # Define hit criteria (multiple levels)
    results_df['Hit_Level'] = 'Not Active'
    
    # Level 3: High confidence hits (all conditions met)
    level3_mask = (
        (results_df['Models_Active_Count'] >= 5) &
        (results_df['Consensus_Avg'] > 0.7) &
        (results_df['Consensus_Weighted'] > 0.7) &
        (results_df['Consensus_Geometric'] > 0.6)
    )
    results_df.loc[level3_mask, 'Hit_Level'] = 'Level_3_High_Confidence'
    
    # Level 2: Medium confidence hits
    level2_mask = (
        (results_df['Models_Active_Count'] >= 4) &
        (results_df['Consensus_Avg'] > 0.6) &
        (results_df['Consensus_Weighted'] > 0.6) &
        (~level3_mask)
    )
    results_df.loc[level2_mask, 'Hit_Level'] = 'Level_2_Medium_Confidence'
    
    # Level 1: Low confidence hits
    level1_mask = (
        (results_df['Models_Active_Count'] >= 3) &
        (results_df['Consensus_Avg'] > 0.5) &
        (~level2_mask) & (~level3_mask)
    )
    results_df.loc[level1_mask, 'Hit_Level'] = 'Level_1_Low_Confidence'
    
    # Calculate hit statistics
    print(f"\nHit Statistics:")
    print(f"  Level 3 (High Confidence): {(results_df['Hit_Level'] == 'Level_3_High_Confidence').sum()}")
    print(f"  Level 2 (Medium Confidence): {(results_df['Hit_Level'] == 'Level_2_Medium_Confidence').sum()}")
    print(f"  Level 1 (Low Confidence): {(results_df['Hit_Level'] == 'Level_1_Low_Confidence').sum()}")
    print(f"  Total Potential Hits: {(results_df['Hit_Level'] != 'Not Active').sum()}")
    
    return results_df

def save_hits_and_predictions(results_df, predictions):
    """
    Save all predictions and hit compounds in various formats
    """
    print("\n" + "="*70)
    print("SAVING RESULTS")
    print("="*70)
    
    # 1. Save complete predictions for all 100,000 compounds
    print("\n1. Saving complete prediction results...")
    
    # Select columns to save
    save_columns = ['zincid', 'SMILES'] + \
                   [col for col in results_df.columns if col not in ['zincid', 'SMILES'] and 'Binary' not in col]
    
    complete_results = results_df[save_columns].copy()
    complete_results.to_csv('5-Hits/predictions/complete_predictions_all_100k.csv', index=False)
    print(f"   ✓ Saved: 5-Hits/predictions/complete_predictions_all_100k.csv ({len(complete_results)} compounds)")
    
    # 2. Save predictions summary statistics
    summary_stats = {}
    for model_name in ['RF_Morgan', 'XGB_Morgan', 'DNN_Morgan', 
                      'RF_MACCS', 'XGB_MACCS', 'DNN_MACCS', 
                      'Consensus_Avg', 'Consensus_Weighted', 'Consensus_Geometric']:
        summary_stats[model_name] = {
            'Mean': results_df[model_name].mean(),
            'Std': results_df[model_name].std(),
            'Median': results_df[model_name].median(),
            '75th_Percentile': results_df[model_name].quantile(0.75),
            '90th_Percentile': results_df[model_name].quantile(0.90),
            '95th_Percentile': results_df[model_name].quantile(0.95),
            'Max': results_df[model_name].max()
        }
    
    summary_df = pd.DataFrame(summary_stats).T
    summary_df.to_csv('5-Hits/predictions/prediction_statistics.csv')
    print(f"   ✓ Saved: 5-Hits/predictions/prediction_statistics.csv")
    
    # 3. Save Level 3 hits (High confidence - 5 models agree)
    level3_hits = results_df[results_df['Hit_Level'] == 'Level_3_High_Confidence'].copy()
    if len(level3_hits) > 0:
        level3_hits = level3_hits.sort_values('Consensus_Weighted', ascending=False)
        level3_hits.to_csv('5-Hits/hits/Level3_High_Confidence_Hits.csv', index=False)
        print(f"\n2. High confidence hits (Level 3): {len(level3_hits)} compounds")
        print(f"   ✓ Saved: 5-Hits/hits/Level3_High_Confidence_Hits.csv")
        
        # Create a simplified hit list for quick review
        level3_simple = level3_hits[['zincid', 'SMILES', 'Consensus_Avg', 'Consensus_Weighted', 
                                      'Models_Active_Count', 'RF_Morgan', 'XGB_Morgan', 'DNN_Morgan',
                                      'RF_MACCS', 'XGB_MACCS', 'DNN_MACCS']].copy()
        level3_simple.to_csv('5-Hits/hits/Level3_Hits_Summary.csv', index=False)
        
        # Save as SDF format for docking (if possible)
        try:
            from rdkit import Chem
            from rdkit.Chem import PandasTools
            
            # Create a simple SDF file
            writer = Chem.SDWriter('5-Hits/hits/Level3_Hits.sdf')
            for idx, row in level3_hits.iterrows():
                mol = Chem.MolFromSmiles(row['SMILES'])
                if mol:
                    mol.SetProp('_Name', str(row['zincid']))
                    mol.SetProp('Consensus_Score', f"{row['Consensus_Weighted']:.4f}")
                    writer.write(mol)
            writer.close()
            print(f"   ✓ Saved: 5-Hits/hits/Level3_Hits.sdf (for docking)")
        except Exception as e:
            print(f"   ⚠ Could not save SDF file: {str(e)[:100]}")
    
    # 4. Save Level 2 hits (Medium confidence - 4 models agree)
    level2_hits = results_df[results_df['Hit_Level'] == 'Level_2_Medium_Confidence'].copy()
    if len(level2_hits) > 0:
        level2_hits = level2_hits.sort_values('Consensus_Weighted', ascending=False)
        level2_hits.to_csv('5-Hits/hits/Level2_Medium_Confidence_Hits.csv', index=False)
        print(f"\n3. Medium confidence hits (Level 2): {len(level2_hits)} compounds")
        print(f"   ✓ Saved: 5-Hits/hits/Level2_Medium_Confidence_Hits.csv")
    
    # 5. Save Level 1 hits (Low confidence - 3 models agree)
    level1_hits = results_df[results_df['Hit_Level'] == 'Level_1_Low_Confidence'].copy()
    if len(level1_hits) > 0:
        level1_hits = level1_hits.sort_values('Consensus_Weighted', ascending=False)
        level1_hits.to_csv('5-Hits/hits/Level1_Low_Confidence_Hits.csv', index=False)
        print(f"\n4. Low confidence hits (Level 1): {len(level1_hits)} compounds")
        print(f"   ✓ Saved: 5-Hits/hits/Level1_Low_Confidence_Hits.csv")
    
    # 6. Save top 100 compounds by consensus score
    top100 = results_df.nlargest(100, 'Consensus_Weighted')[['zincid', 'SMILES', 'Consensus_Weighted', 
                                                               'Models_Active_Count', 'RF_Morgan', 
                                                               'XGB_Morgan', 'DNN_Morgan']]
    top100.to_csv('5-Hits/hits/Top100_Compounds_By_Consensus.csv', index=False)
    print(f"\n5. Top 100 compounds by weighted consensus: {len(top100)} compounds")
    print(f"   ✓ Saved: 5-Hits/hits/Top100_Compounds_By_Consensus.csv")
    
    return level3_hits, level2_hits, level1_hits

def generate_hit_report(results_df, level3_hits, level2_hits, level1_hits):
    """
    Generate a comprehensive hit report for publication
    """
    print("\n" + "="*70)
    print("GENERATING HIT REPORT")
    print("="*70)
    
    report_lines = []
    report_lines.append("="*70)
    report_lines.append("VIRTUAL SCREENING HIT REPORT")
    report_lines.append(f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append("="*70)
    report_lines.append("")
    report_lines.append("SCREENING SUMMARY:")
    report_lines.append(f"  Total compounds screened: 100,000")
    report_lines.append(f"  Compounds with valid fingerprints: {len(results_df)}")
    report_lines.append("")
    report_lines.append("HIT STATISTICS:")
    report_lines.append(f"  Level 3 (High Confidence - >=5 models): {len(level3_hits)} compounds ({len(level3_hits)/100000*100:.2f}%)")
    report_lines.append(f"  Level 2 (Medium Confidence - >=4 models): {len(level2_hits)} compounds ({len(level2_hits)/100000*100:.2f}%)")
    report_lines.append(f"  Level 1 (Low Confidence - >=3 models): {len(level1_hits)} compounds ({len(level1_hits)/100000*100:.2f}%)")
    report_lines.append(f"  Total potential hits: {len(level3_hits)+len(level2_hits)+len(level1_hits)} ({(len(level3_hits)+len(level2_hits)+len(level1_hits))/100000*100:.2f}%)")
    report_lines.append("")
    
    if len(level3_hits) > 0:
        report_lines.append("TOP 10 HIGH CONFIDENCE HITS:")
        report_lines.append("-"*50)
        top10 = level3_hits.nlargest(10, 'Consensus_Weighted')[['zincid', 'Consensus_Weighted', 'Models_Active_Count', 
                                                                  'RF_Morgan', 'XGB_Morgan', 'DNN_Morgan']]
        for idx, row in top10.iterrows():
            report_lines.append(f"  {row['zincid']}: Consensus={row['Consensus_Weighted']:.4f}, "
                              f"Models Active={int(row['Models_Active_Count'])}/6")
    
    report_lines.append("")
    report_lines.append("MODEL PERFORMANCE ON SCREENING SET:")
    for model in ['RF_Morgan', 'XGB_Morgan', 'DNN_Morgan', 'RF_MACCS', 'XGB_MACCS', 'DNN_MACCS']:
        active_percent = (results_df[model] > 0.5).mean() * 100
        high_conf_percent = (results_df[model] > 0.8).mean() * 100
        report_lines.append(f"  {model}: {active_percent:.1f}% active (>0.5), {high_conf_percent:.1f}% high confidence (>0.8)")
    
    report_lines.append("")
    report_lines.append("RECOMMENDATIONS FOR EXPERIMENTAL VALIDATION:")
    report_lines.append(f"  1. Prioritize Level 3 hits (n={len(level3_hits)}) for in vitro testing")
    report_lines.append("  2. Perform dose-response IC50 assays on top 20-30 compounds")
    report_lines.append("  3. Verify selectivity against other lactate dehydrogenase isoforms")
    report_lines.append("  4. Assess cytotoxicity in colon cancer cell lines (e.g., HCT116, HT29)")
    report_lines.append("  5. Evaluate pharmacokinetic properties of most potent hits")
    report_lines.append("")
    report_lines.append("="*70)
    
    # Save report with UTF-8 encoding to avoid Unicode errors
    with open('5-Hits/HIT_REPORT.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(report_lines))
    
    print("\n✓ Hit report saved: 5-Hits/HIT_REPORT.txt")
    
    # Print report to console with error handling for special characters
    for line in report_lines:
        try:
            print(line)
        except:
            # Replace problematic characters
            line_clean = line.encode('ascii', 'ignore').decode('ascii')
            print(line_clean)

# ============================================================================
# MAIN SCREENING PIPELINE
# ============================================================================

def main():
    print("="*70)
    print("VIRTUAL SCREENING PIPELINE FOR LDHA INHIBITORS")
    print("="*70)
    print("\nGoal: Screen 100,000 compounds to identify potential LDHA inhibitors")
    print("Strategy: Consensus scoring using 6 trained models (3 models x 2 fingerprints)")
    
    # 1. Load fingerprints
    print("\n" + "-"*50)
    print("STEP 1: Loading screening fingerprints")
    print("-"*50)
    
    morgan_fps = np.load('data/screening_fingerprints/screening_morgan_fps.npy')
    maccs_fps = np.load('data/screening_fingerprints/screening_maccs_fps.npy')
    metadata = pd.read_csv('data/screening_fingerprints/screening_morgan_metadata.csv')
    
    print(f"  Morgan fingerprints: {morgan_fps.shape}")
    print(f"  MACCS fingerprints: {maccs_fps.shape}")
    print(f"  Metadata: {len(metadata)} compounds")
    
    # 2. Load models
    print("\n" + "-"*50)
    print("STEP 2: Loading trained models")
    print("-"*50)
    models, scaler_morgan, scaler_maccs = load_models_and_scalers()
    
    # 3. Make predictions
    print("\n" + "-"*50)
    print("STEP 3: Making predictions with all models")
    print("-"*50)
    predictions = predict_with_consensus(models, scaler_morgan, scaler_maccs, 
                                        morgan_fps, maccs_fps, metadata)
    
    # 4. Identify hits
    print("\n" + "-"*50)
    print("STEP 4: Identifying high-confidence hits")
    print("-"*50)
    
    thresholds = {
        'consensus_avg': 0.6,
        'consensus_weighted': 0.6,
        'min_votes': 4
    }
    
    results_df = identify_hits(predictions, metadata, thresholds)
    
    # 5. Save results
    print("\n" + "-"*50)
    print("STEP 5: Saving results and hit lists")
    print("-"*50)
    level3_hits, level2_hits, level1_hits = save_hits_and_predictions(results_df, predictions)
    
    # 6. Generate report
    print("\n" + "-"*50)
    print("STEP 6: Generating hit report")
    print("-"*50)
    generate_hit_report(results_df, level3_hits, level2_hits, level1_hits)
    
    # 7. Final summary
    print("\n" + "="*70)
    print("SCREENING COMPLETED SUCCESSFULLY!")
    print("="*70)
    print("\nOutput files saved in '5-Hits/' directory:")
    print("  predictions/ - Complete predictions for all 100,000 compounds")
    print("  hits/ - Hit lists at different confidence levels")
    print("  HIT_REPORT.txt - Detailed screening report")
    
    if len(level3_hits) > 0:
        print(f"\nRECOMMENDATION: Prioritize {len(level3_hits)} Level 3 hits for experimental validation")
        print(f"  Top hit: {level3_hits.iloc[0]['zincid']} (Consensus score: {level3_hits.iloc[0]['Consensus_Weighted']:.4f})")
    else:
        print("\nNo high-confidence hits found. Consider lowering thresholds or reviewing models.")
    
    # Display top 5 hits
    if len(level3_hits) > 0:
        print("\n" + "-"*50)
        print("TOP 5 HIGH CONFIDENCE HITS:")
        print("-"*50)
        for i in range(min(5, len(level3_hits))):
            hit = level3_hits.iloc[i]
            print(f"\n  {i+1}. ZINC ID: {hit['zincid']}")
            print(f"     Consensus Score: {hit['Consensus_Weighted']:.4f}")
            print(f"     Models Active: {int(hit['Models_Active_Count'])}/6")
            print(f"     RF_Morgan: {hit['RF_Morgan']:.4f}, XGB_Morgan: {hit['XGB_Morgan']:.4f}, DNN_Morgan: {hit['DNN_Morgan']:.4f}")
            print(f"     SMILES: {hit['SMILES'][:100]}...")
    
    print("\n✅ Virtual screening pipeline complete!")

# Run the screening pipeline
if __name__ == "__main__":
    main()

VIRTUAL SCREENING PIPELINE FOR LDHA INHIBITORS

Goal: Screen 100,000 compounds to identify potential LDHA inhibitors
Strategy: Consensus scoring using 6 trained models (3 models x 2 fingerprints)

--------------------------------------------------
STEP 1: Loading screening fingerprints
--------------------------------------------------
  Morgan fingerprints: (100000, 2048)
  MACCS fingerprints: (100000, 167)
  Metadata: 100000 compounds

--------------------------------------------------
STEP 2: Loading trained models
--------------------------------------------------
Loading trained models...
  ✓ All 6 models and scalers loaded successfully

--------------------------------------------------
STEP 3: Making predictions with all models
--------------------------------------------------

Performing predictions with all 6 models...
  Scaling fingerprints...
  Random Forest (Morgan)...
  Random Forest (MACCS)...
  XGBoost (Morgan)...
  XGBoost (MACCS)...
  DNN (Morgan)...
  DNN (MACCS)...


In [19]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolAlign
from rdkit.Chem import Draw
import os
from openbabel import openbabel
import warnings
warnings.filterwarnings('ignore')

# Create directories for PDB files
os.makedirs('5-Hits/pdb_structures', exist_ok=True)
os.makedirs('5-Hits/pdb_structures/individual_pdb', exist_ok=True)
os.makedirs('5-Hits/pdb_structures/combined_structures', exist_ok=True)
os.makedirs('5-Hits/pdb_structures/2d_images', exist_ok=True)
os.makedirs('5-Hits/pdb_structures/3d_visualizations', exist_ok=True)

# ============================================================================
# FUNCTION 1: Generate 3D Structure from SMILES using RDKit
# ============================================================================

def generate_3d_structure_rdkit(smiles, compound_id, output_dir, force_field='MMFF'):
    """
    Generate 3D PDB structure using RDKit with multiple force fields
    
    Parameters:
    - smiles: SMILES string
    - compound_id: ZINC ID or compound identifier
    - output_dir: Directory to save PDB file
    - force_field: 'MMFF', 'UFF', or 'ETKDG'
    
    Returns:
    - success: Boolean indicating success
    - pdb_path: Path to saved PDB file
    - mol: RDKit mol object
    """
    
    try:
        # Convert SMILES to mol object
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"  ✗ Failed to parse SMILES for {compound_id}")
            return False, None, None
        
        # Add hydrogens
        mol = Chem.AddHs(mol)
        
        # Generate 3D coordinates
        if force_field == 'ETKDG':
            # ETKDG method for embedding (faster, good for initial structure)
            AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
            AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
        else:
            # Standard embedding
            AllChem.EmbedMolecule(mol, randomSeed=42)
        
        # Optimize geometry with MMFF force field
        if force_field == 'MMFF':
            AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
        elif force_field == 'UFF':
            AllChem.UFFOptimizeMolecule(mol, maxIters=500)
        
        # Generate a cleaner conformation
        AllChem.AlignMol(mol, mol)
        
        # Save as PDB file
        pdb_filename = f"{compound_id}.pdb"
        pdb_path = os.path.join(output_dir, pdb_filename)
        Chem.MolToPDBFile(mol, pdb_path)
        
        # Calculate molecular properties
        properties = {
            'MW': Descriptors.ExactMolWt(mol),
            'LogP': Descriptors.MolLogP(mol),
            'HBA': Descriptors.NumHAcceptors(mol),
            'HBD': Descriptors.NumHDonors(mol),
            'RotBonds': Descriptors.NumRotatableBonds(mol),
            'TPSA': Descriptors.TPSA(mol),
            'HeavyAtoms': mol.GetNumHeavyAtoms()
        }
        
        return True, pdb_path, mol, properties
        
    except Exception as e:
        print(f"  ✗ Error for {compound_id}: {str(e)[:100]}")
        return False, None, None, None

# ============================================================================
# FUNCTION 2: Generate PDB using OpenBabel (Alternative method)
# ============================================================================

def generate_3d_structure_openbabel(smiles, compound_id, output_dir):
    """
    Generate 3D structure using OpenBabel (alternative method)
    """
    try:
        obConversion = openbabel.OBConversion()
        obConversion.SetInAndOutFormats("smi", "pdb")
        
        mol = openbabel.OBMol()
        obConversion.ReadString(mol, smiles)
        
        # Generate 3D coordinates
        mol.AddHydrogens()
        mol.SetSpinMultiplicity(1)
        mol.SetTotalCharge(0)
        
        # Optimize geometry
        ff = openbabel.OBForceField.FindForceField("MMFF94")
        if ff:
            ff.Setup(mol)
            ff.ConjugateGradients(500)
            ff.GetCoordinates(mol)
        
        # Save PDB file
        pdb_filename = f"{compound_id}_obabel.pdb"
        pdb_path = os.path.join(output_dir, pdb_filename)
        obConversion.WriteFile(mol, pdb_path)
        
        return True, pdb_path
        
    except Exception as e:
        print(f"  ✗ OpenBabel error for {compound_id}: {str(e)[:100]}")
        return False, None

# ============================================================================
# FUNCTION 3: Generate 2D Image of Compound
# ============================================================================

def generate_2d_image(smiles, compound_id, output_dir, img_size=(800, 600)):
    """
    Generate 2D structure image using RDKit
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            img = Draw.MolToImage(mol, size=img_size)
            img_path = os.path.join(output_dir, f"{compound_id}_2d.png")
            img.save(img_path)
            return True, img_path
    except Exception as e:
        print(f"  ✗ 2D image error for {compound_id}: {str(e)[:50]}")
        return False, None

# ============================================================================
# FUNCTION 4: Generate Multiple Conformations
# ============================================================================

def generate_conformers(smiles, compound_id, output_dir, n_conformers=5):
    """
    Generate multiple conformations for a compound
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        mol = Chem.AddHs(mol)
        
        # Embed multiple conformers
        cids = AllChem.EmbedMultipleConfs(mol, numConfs=n_conformers, 
                                         randomSeed=42,
                                         numThreads=4)
        
        # Optimize each conformer
        for cid in cids:
            AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=200)
        
        # Save all conformers in a single PDB file
        conformer_dir = os.path.join(output_dir, 'conformers')
        os.makedirs(conformer_dir, exist_ok=True)
        
        pdb_path = os.path.join(conformer_dir, f"{compound_id}_conformers.pdb")
        
        # Write multi-model PDB
        with open(pdb_path, 'w') as f:
            for i, cid in enumerate(cids):
                f.write(f"MODEL     {i+1}\n")
                pdb_block = Chem.MolToPDBBlock(mol, confId=cid)
                f.write(pdb_block)
                f.write("ENDMDL\n")
        
        return True, pdb_path, len(cids)
        
    except Exception as e:
        print(f"  ✗ Conformer generation error: {str(e)[:100]}")
        return False, None, 0

# ============================================================================
# FUNCTION 5: Create Combined PDB File
# ============================================================================

def create_combined_pdb(all_molecules, output_path):
    """
    Combine multiple molecules into a single PDB file
    """
    try:
        with open(output_path, 'w') as f:
            f.write("REMARK Combined PDB file for all 26 high-confidence hits\n")
            f.write("REMARK Generated by RDKit\n")
            f.write("REMARK Date: " + pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S') + "\n")
            f.write("REMARK Number of compounds: " + str(len(all_molecules)) + "\n\n")
            
            for idx, (compound_id, mol) in enumerate(all_molecules):
                f.write(f"REMARK Compound {idx+1}: {compound_id}\n")
                pdb_block = Chem.MolToPDBBlock(mol)
                # Add TER card between molecules
                f.write(pdb_block)
                f.write("TER\n")
        
        return True
    except Exception as e:
        print(f"  ✗ Error creating combined PDB: {str(e)}")
        return False

# ============================================================================
# FUNCTION 6: Validate PDB Structure Quality
# ============================================================================

def validate_pdb_structure(pdb_path):
    """
    Validate PDB file quality (check for missing atoms, clashes)
    """
    try:
        mol = Chem.MolFromPDBFile(pdb_path, removeHs=False)
        if mol is None:
            return False, "Failed to read PDB"
        
        # Check for issues
        issues = []
        if mol.GetNumAtoms() == 0:
            issues.append("No atoms found")
        
        # Check bond lengths (simplified validation)
        bond_lengths = []
        for bond in mol.GetBonds():
            begin_atom = bond.GetBeginAtom()
            end_atom = bond.GetEndAtom()
            pos1 = mol.GetConformer().GetAtomPosition(begin_atom.GetIdx())
            pos2 = mol.GetConformer().GetAtomPosition(end_atom.GetIdx())
            dist = (pos1.x - pos2.x)**2 + (pos1.y - pos2.y)**2 + (pos1.z - pos2.z)**2
            dist = dist**0.5
            bond_lengths.append(dist)
        
        if bond_lengths:
            avg_bond = np.mean(bond_lengths)
            if avg_bond < 0.8 or avg_bond > 1.8:
                issues.append(f"Unusual bond lengths: {avg_bond:.2f} Å")
        
        return len(issues) == 0, "; ".join(issues) if issues else "Valid structure"
        
    except Exception as e:
        return False, str(e)

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print("="*70)
    print("3D PDB STRUCTURE GENERATION FOR 26 HIGH-CONFIDENCE HITS")
    print("="*70)
    
    # Load high-confidence hits
    hits_df = pd.read_csv('5-Hits/hits/Level3_High_Confidence_Hits.csv')
    
    print(f"\nLoaded {len(hits_df)} high-confidence compounds")
    print(f"Columns available: {hits_df.columns.tolist()}\n")
    
    # Prepare results storage
    results = []
    all_molecules = []
    
    # Process each compound
    for idx, row in hits_df.iterrows():
        compound_id = row['zincid']
        smiles = row['SMILES']
        consensus_score = row.get('Consensus_Weighted', row.get('Consensus_Avg', 0))
        
        print(f"\n[{idx+1}/{len(hits_df)}] Processing: {compound_id}")
        print(f"  Consensus Score: {consensus_score:.4f}")
        
        # Create compound-specific directory
        compound_dir = os.path.join('5-Hits/pdb_structures/individual_pdb', compound_id)
        os.makedirs(compound_dir, exist_ok=True)
        
        # Generate 3D structure with RDKit (MMFF)
        success_rdkit, pdb_path_rdkit, mol, props = generate_3d_structure_rdkit(
            smiles, compound_id, compound_dir, force_field='MMFF'
        )
        
        if success_rdkit and mol:
            # Validate structure
            valid, validation_msg = validate_pdb_structure(pdb_path_rdkit)
            
            # Generate 2D image
            success_img, img_path = generate_2d_image(
                smiles, compound_id, compound_dir, img_size=(800, 600)
            )
            
            # Generate conformers (optional, for flexibility)
            success_conf, conf_path, n_confs = generate_conformers(
                smiles, compound_id, compound_dir, n_conformers=5
            )
            
            # Store for combined file
            all_molecules.append((compound_id, mol))
            
            # Record results
            results.append({
                'Compound_ID': compound_id,
                'SMILES': smiles,
                'Consensus_Score': consensus_score,
                'PDB_Generated': True,
                'PDB_Path': pdb_path_rdkit,
                'Validation': validation_msg,
                '2D_Image': img_path if success_img else None,
                'Conformers_Generated': n_confs if success_conf else 0,
                'Molecular_Weight': props['MW'],
                'LogP': props['LogP'],
                'HBA': props['HBA'],
                'HBD': props['HBD'],
                'Rotatable_Bonds': props['RotBonds'],
                'TPSA': props['TPSA'],
                'Heavy_Atoms': props['HeavyAtoms']
            })
            
            print(f"  ✓ PDB generated: {pdb_path_rdkit}")
            print(f"  ✓ Validation: {validation_msg}")
            if success_img:
                print(f"  ✓ 2D image generated")
            if success_conf:
                print(f"  ✓ {n_confs} conformers generated")
            
        else:
            # Try alternative method with OpenBabel
            success_obabel, pdb_path_obabel = generate_3d_structure_openbabel(
                smiles, compound_id, compound_dir
            )
            
            if success_obabel:
                results.append({
                    'Compound_ID': compound_id,
                    'SMILES': smiles,
                    'Consensus_Score': consensus_score,
                    'PDB_Generated': True,
                    'PDB_Path': pdb_path_obabel,
                    'Validation': 'Generated with OpenBabel',
                    '2D_Image': None,
                    'Conformers_Generated': 0,
                    'Molecular_Weight': None,
                    'LogP': None,
                    'HBA': None,
                    'HBD': None,
                    'Rotatable_Bonds': None,
                    'TPSA': None,
                    'Heavy_Atoms': None
                })
                print(f"  ✓ PDB generated with OpenBabel: {pdb_path_obabel}")
            else:
                results.append({
                    'Compound_ID': compound_id,
                    'SMILES': smiles,
                    'Consensus_Score': consensus_score,
                    'PDB_Generated': False,
                    'PDB_Path': None,
                    'Validation': 'Failed',
                    '2D_Image': None,
                    'Conformers_Generated': 0,
                    'Molecular_Weight': None,
                    'LogP': None,
                    'HBA': None,
                    'HBD': None,
                    'Rotatable_Bonds': None,
                    'TPSA': None,
                    'Heavy_Atoms': None
                })
                print(f"  ✗ Failed to generate PDB")
    
    # Create combined PDB file
    if all_molecules:
        combined_path = '5-Hits/pdb_structures/combined_structures/all_26_hits_combined.pdb'
        if create_combined_pdb(all_molecules, combined_path):
            print(f"\n✓ Combined PDB created: {combined_path}")
    
    # Save results summary
    results_df = pd.DataFrame(results)
    results_df.to_csv('5-Hits/pdb_structures/pdb_generation_summary.csv', index=False)
    
    # Generate summary report
    print("\n" + "="*70)
    print("PDB GENERATION SUMMARY")
    print("="*70)
    
    successful = results_df['PDB_Generated'].sum()
    total = len(results_df)
    
    print(f"\nTotal compounds processed: {total}")
    print(f"Successfully generated: {successful}/{total} ({successful/total*100:.1f}%)")
    print(f"Failed: {total-successful}/{total}")
    
    if successful > 0:
        print(f"\nMolecular Properties of Generated Structures (Mean ± SD):")
        print(f"  Molecular Weight: {results_df['Molecular_Weight'].dropna().mean():.1f} ± {results_df['Molecular_Weight'].dropna().std():.1f} Da")
        print(f"  LogP: {results_df['LogP'].dropna().mean():.2f} ± {results_df['LogP'].dropna().std():.2f}")
        print(f"  HBA: {results_df['HBA'].dropna().mean():.1f} ± {results_df['HBA'].dropna().std():.1f}")
        print(f"  HBD: {results_df['HBD'].dropna().mean():.1f} ± {results_df['HBD'].dropna().std():.1f}")
        print(f"  TPSA: {results_df['TPSA'].dropna().mean():.1f} ± {results_df['TPSA'].dropna().std():.1f} Å²")
    
    # Generate a summary text file
    with open('5-Hits/pdb_structures/README.txt', 'w') as f:
        f.write("="*70 + "\n")
        f.write("PDB STRUCTURES FOR 26 HIGH-CONFIDENCE LDHA INHIBITOR CANDIDATES\n")
        f.write("="*70 + "\n\n")
        f.write(f"Date generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total compounds: {total}\n")
        f.write(f"Successfully generated: {successful}\n\n")
        
        f.write("DIRECTORY STRUCTURE:\n")
        f.write("- individual_pdb/: Individual PDB files for each compound\n")
        f.write("- combined_structures/: Combined PDB file with all compounds\n")
        f.write("- 2d_images/: 2D structure images\n")
        f.write("- 3d_visualizations/: 3D visualization files\n\n")
        
        f.write("FILE NAMING CONVENTION:\n")
        f.write("- {ZINC_ID}.pdb: 3D structure optimized with MMFF94 force field\n")
        f.write("- {ZINC_ID}_2d.png: 2D structure image\n")
        f.write("- {ZINC_ID}_conformers.pdb: Multiple conformations (5 per compound)\n\n")
        
        f.write("SUCCESSFULLY GENERATED COMPOUNDS:\n")
        for idx, row in results_df[results_df['PDB_Generated']].iterrows():
            f.write(f"  {row['Compound_ID']}: {row['Consensus_Score']:.4f}\n")
        
        if (total-successful) > 0:
            f.write("\nFAILED COMPOUNDS:\n")
            for idx, row in results_df[~results_df['PDB_Generated']].iterrows():
                f.write(f"  {row['Compound_ID']}\n")
    
    print(f"\n✓ Summary saved: 5-Hits/pdb_structures/pdb_generation_summary.csv")
    print(f"✓ README saved: 5-Hits/pdb_structures/README.txt")
    
    # Display successful compounds
    print("\n" + "-"*50)
    print("SUCCESSFULLY GENERATED PDB FILES:")
    print("-"*50)
    for idx, row in results_df[results_df['PDB_Generated']].head(10).iterrows():
        print(f"  ✓ {row['Compound_ID']}")
    
    if successful > 10:
        print(f"  ... and {successful-10} more compounds")
    
    print("\n" + "="*70)
    print("✅ PDB STRUCTURE GENERATION COMPLETE!")
    print("="*70)
    print("\nOutput directory: 5-Hits/pdb_structures/")
    print("  - Use individual PDB files for docking studies")
    print("  - Use combined PDB file for visualization of all hits")
    print("  - 2D images available for quick structural review")
    print("  - Multiple conformers available for flexible docking")

# Run the main function
if __name__ == "__main__":
    main()

3D PDB STRUCTURE GENERATION FOR 26 HIGH-CONFIDENCE HITS

Loaded 26 high-confidence compounds
Columns available: ['zincid', 'SMILES', 'RF_Morgan', 'RF_MACCS', 'XGB_Morgan', 'XGB_MACCS', 'DNN_Morgan', 'DNN_MACCS', 'Consensus_Avg', 'Consensus_Weighted', 'Consensus_Votes', 'Consensus_Geometric', 'RF_Morgan_Binary', 'XGB_Morgan_Binary', 'DNN_Morgan_Binary', 'RF_MACCS_Binary', 'XGB_MACCS_Binary', 'DNN_MACCS_Binary', 'Models_Active_Count', 'Consensus_Active', 'Consensus_Weighted_Active', 'Vote_Active', 'Hit_Level']


[1/26] Processing: ZINCsN00000gCsav
  Consensus Score: 0.7617
  ✓ PDB generated: 5-Hits/pdb_structures/individual_pdb\ZINCsN00000gCsav\ZINCsN00000gCsav.pdb
  ✓ Validation: Valid structure
  ✓ 2D image generated
  ✓ 5 conformers generated

[2/26] Processing: ZINCsj00000oQSX6
  Consensus Score: 0.7608
  ✓ PDB generated: 5-Hits/pdb_structures/individual_pdb\ZINCsj00000oQSX6\ZINCsj00000oQSX6.pdb
  ✓ Validation: Valid structure
  ✓ 2D image generated
  ✓ 5 conformers generated

[3/26]

In [20]:
# Generate PyMOL visualization script for all 26 hits

def generate_pymol_script():
    """
    Generate a PyMOL script to visualize all 26 hits
    """
    
    pymol_script = """
# PyMOL Visualization Script for 26 LDHA Inhibitor Candidates
# Generated by RDKit-based pipeline

# Load all compounds
load 5-Hits/pdb_structures/combined_structures/all_26_hits_combined.pdb

# Set visualization style
hide everything
show cartoon, all
show sticks, all
set stick_radius, 0.15
set sphere_scale, 0.3
set bg_color, white

# Color by element
util.cbaw

# Align structures for comparison
align all

# Create a grid layout
viewport 1200, 800
set grid_mode, 1

# Label compounds
for i in range(1, 27):
    label resi i, f"Hit_{i}"
    
# Save session
save 5-Hits/pdb_structures/combined_structures/all_hits_visualization.pse

# Export image
png 5-Hits/pdb_structures/combined_structures/all_hits_overview.png, dpi=300, ray=1

print("PyMOL visualization saved successfully!")
"""
    
    with open('5-Hits/pdb_structures/visualize_hits.pml', 'w') as f:
        f.write(pymol_script)
    
    print("✓ PyMOL visualization script saved: 5-Hits/pdb_structures/visualize_hits.pml")
    print("  Run with: pymol visualize_hits.pml")

# Generate the PyMOL script
generate_pymol_script()

✓ PyMOL visualization script saved: 5-Hits/pdb_structures/visualize_hits.pml
  Run with: pymol visualize_hits.pml


In [22]:
from rdkit import Chem
from rdkit.Chem import Draw
import pandas as pd
from PIL import Image
import io

# Load hits
hits = pd.read_csv('5-Hits/hits/Level3_High_Confidence_Hits.csv')

# Create high-quality images for each compound
mols = []
legends = []
for idx, row in hits.iterrows():
    mol = Chem.MolFromSmiles(row['SMILES'])
    if mol:
        # Generate high-res individual image
        img_single = Draw.MolToImage(mol, size=(400, 400))
        mols.append(mol)
        legends.append(f"{row['zincid']}\n{row['Consensus_Weighted']:.3f}")

# Create grid with higher resolution
img = Draw.MolsToGridImage(mols, 
                          molsPerRow=5, 
                          subImgSize=(400, 400),
                          legends=legends,
                          useSVG=False)

# Convert to PIL Image and save as high-quality PNG
if hasattr(img, 'save'):
    img.save('5-Hits/hits/All_26_Hits_Grid.png', dpi=(300, 300), quality=95)
else:
    # Handle different return types
    pil_img = Image.open(io.BytesIO(img.data))
    pil_img.save('5-Hits/hits/All_26_Hits_Grid.png', dpi=(300, 300), quality=95)

print(f"✓ High-resolution grid image saved: 5-Hits/hits/All_26_Hits_Grid.png")
print(f"  Resolution: 300 DPI (publication quality)")

✓ High-resolution grid image saved: 5-Hits/hits/All_26_Hits_Grid.png
  Resolution: 300 DPI (publication quality)
